# El amplificador operacional en acción · Semana 3
**Circuitos eléctricos y microcontroladores**

Ejemplos prácticos con esquemas, ecuaciones de diseño y simulaciones:

1. **Lazo abierto:** comparador con referencia, comparador frente a Schmitt y **semáforo hecho solo con operacionales** (diseño paso a paso).
2. **Realimentación negativa:** sumador (convertidor D/A de 3 bits), restador diferencial con shunt y amplificador multientradas universal.
3. **Realimentación positiva:** termostato con histéresis (ON/OFF frente a control P), juego de reflejos con biestable y theremin óptico.
4. **Página web** con todos los ejemplos funcionando, desplegada dentro de este cuaderno.

> **Cómo usarlo:** *Entorno de ejecución → Ejecutar todas*. No hay que instalar nada: los esquemas se dibujan con una pequeña biblioteca hecha con `matplotlib` (celda 2). Las celdas con controles deslizantes usan `ipywidgets`, que ya viene en Colab.

**Modelo de trabajo:** TL084 a ±12 V, $V_{OH} = +10{,}5\ \mathrm{V}$, $V_{OL} = -10{,}5\ \mathrm{V}$, slew rate $13\ \mathrm{V/\mu s}$, diodo 1N4148 con $V_D = 0{,}6\ \mathrm{V}$.

In [ ]:
# Celda 1 · Configuración general
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['mathtext.fontset'] = 'dejavusans'
plt.rcParams['figure.dpi'] = 110

VOH, VOL, VD = 10.5, -10.5, 0.6     # saturación del TL084 a ±12 V y caída del 1N4148

def coma(x, d=2):
    """Formatea un número con coma decimal."""
    return f'{x:.{d}f}'.replace('.', ',')

def paralelo(*R):
    return 1 / sum(1 / r for r in R)

def interactivo(f, **controles):
    """Usa ipywidgets si está disponible; si no, ejecuta la función con los valores por defecto."""
    try:
        from ipywidgets import interact
        return interact(f, **controles)
    except Exception:
        return f(**{k: (v.value if hasattr(v, 'value') else v) for k, v in controles.items()})

def deslizador(valor, minimo, maximo, paso, nombre):
    try:
        from ipywidgets import FloatSlider, Layout
        return FloatSlider(value=valor, min=minimo, max=maximo, step=paso, description=nombre,
                           continuous_update=False, style={'description_width': '140px'}, layout=Layout(width='520px'))
    except Exception:
        return valor
print('Listo')

In [ ]:
# Celda 2 · Biblioteca de esquemas electrónicos (solo matplotlib)
"""Mini biblioteca para dibujar esquemas electrónicos con matplotlib.
No necesita instalar nada más que matplotlib y numpy (funciona igual en Google Colab).
Coordenadas en "unidades de rejilla"; cada componente se dibuja entre dos puntos alineados."""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Circle, FancyBboxPatch

INK = '#1F2A33'
LED_COLORS = {'rojo': '#D2402F', 'amarillo': '#E8A317', 'verde': '#2E9B57', 'azul': '#2F6DB5'}


class Esquema:
    def __init__(self, w, h, escala=0.62, fondo='white'):
        self.w, self.h = w, h
        self.fig = plt.figure(figsize=(w * escala, h * escala), facecolor=fondo)
        self.ax = self.fig.add_axes([0, 0, 1, 1])
        self.ax.set_xlim(0, w)
        self.ax.set_ylim(0, h)
        self.ax.set_aspect('equal')
        self.ax.axis('off')
        self.lw = 2.0

    # ---------- primitivas ----------
    def linea(self, *pts, color=INK, lw=None, ls='-'):
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        self.ax.plot(xs, ys, color=color, lw=lw or self.lw, ls=ls, solid_capstyle='round', solid_joinstyle='round')

    def linea_saltos(self, p1, p2, saltos=(), r=0.16, color=INK):
        """Línea horizontal de p1 a p2 que salta (semicírculo) sobre los cruces en las x indicadas."""
        y = p1[1]
        xs = sorted(saltos) if p2[0] > p1[0] else sorted(saltos, reverse=True)
        cur = p1[0]
        sgn = 1 if p2[0] > p1[0] else -1
        for xc in xs:
            self.linea((cur, y), (xc - sgn * r, y), color=color)
            t = np.linspace(np.pi, 0, 30) if sgn > 0 else np.linspace(0, np.pi, 30)
            self.ax.plot(xc + r * np.cos(t), y + r * np.sin(t), color=color, lw=self.lw)
            cur = xc + sgn * r
        self.linea((cur, y), (p2[0], y), color=color)

    def nodo(self, p):
        self.ax.add_patch(Circle(p, 0.09, color=INK, zorder=5))

    def texto(self, p, t, ha='center', va='center', size=13, color=INK, weight='normal'):
        self.ax.text(p[0], p[1], t, ha=ha, va=va, fontsize=size, color=color, fontweight=weight, zorder=6)

    def _eje(self, p1, p2):
        p1, p2 = np.array(p1, float), np.array(p2, float)
        d = p2 - p1
        L = np.hypot(*d)
        u = d / L
        n = np.array([-u[1], u[0]])
        return p1, p2, L, u, n

    def _etiqueta(self, centro, n, label, lado, dist=0.45, size=13):
        if not label:
            return
        c = centro + n * dist * lado
        ha = 'center'
        if abs(n[0]) > 0.5:
            ha = 'left' if n[0] * lado > 0 else 'right'
        va = 'center'
        if abs(n[1]) > 0.5:
            va = 'bottom' if n[1] * lado > 0 else 'top'
        self.texto(c, label, ha=ha, va=va, size=size)

    def resistencia(self, p1, p2, label=None, lado=1, cuerpo=0.9, color=INK):
        p1, p2, L, u, n = self._eje(p1, p2)
        c = (p1 + p2) / 2
        a = c - u * cuerpo / 2
        b = c + u * cuerpo / 2
        pts = [p1, a]
        k = 6
        for i in range(k):
            t = (i + 0.5) / k
            pts.append(a + u * cuerpo * t + n * 0.18 * (1 if i % 2 == 0 else -1))
        pts += [b, p2]
        self.linea(*[tuple(p) for p in pts], color=color)
        self._etiqueta(c, n, label, lado)

    def ldr(self, p1, p2, label=None, lado=1):
        self.resistencia(p1, p2, None, lado)
        p1, p2, L, u, n = self._eje(p1, p2)
        c = (p1 + p2) / 2
        self.ax.add_patch(Circle(c, 0.5, fill=False, color=INK, lw=1.4))
        for k in (-0.15, 0.15):
            s = c - n * 0.95 + u * (k - 0.35)
            e = c - n * 0.55 + u * (k - 0.05)
            self.ax.annotate('', xy=e, xytext=s, arrowprops=dict(arrowstyle='->', color=LED_COLORS['amarillo'], lw=1.6))
        self._etiqueta(c, n, label, lado, dist=0.65)

    def condensador(self, p1, p2, label=None, lado=1, polar=False):
        p1, p2, L, u, n = self._eje(p1, p2)
        c = (p1 + p2) / 2
        g = 0.09
        self.linea(tuple(p1), tuple(c - u * g))
        self.linea(tuple(c + u * g), tuple(p2))
        self.linea(tuple(c - u * g + n * 0.32), tuple(c - u * g - n * 0.32), lw=2.6)
        self.linea(tuple(c + u * g + n * 0.32), tuple(c + u * g - n * 0.32), lw=2.6)
        if polar:
            self.texto(tuple(c - u * 0.3 + n * 0.32), '+', size=11)
        self._etiqueta(c, n, label, lado, dist=0.5)

    def diodo(self, p1, p2, label=None, lado=1, led=None):
        """Ánodo en p1, cátodo en p2. led='rojo'|'amarillo'|'verde' dibuja un LED."""
        p1, p2, L, u, n = self._eje(p1, p2)
        c = (p1 + p2) / 2
        a = c - u * 0.25
        k = c + u * 0.25
        self.linea(tuple(p1), tuple(a))
        self.linea(tuple(k), tuple(p2))
        fill = LED_COLORS.get(led, INK) if led else INK
        self.ax.add_patch(Polygon([a + n * 0.28, a - n * 0.28, k], closed=True, fc=fill, ec=INK, lw=1.5, zorder=4))
        self.linea(tuple(k + n * 0.28), tuple(k - n * 0.28), lw=2.4)
        if led:
            for off in (-0.08, 0.17):
                s = c + n * 0.35 + u * off
                e = s + n * 0.35 + u * 0.2
                self.ax.annotate('', xy=e, xytext=s, arrowprops=dict(arrowstyle='->', color=fill, lw=1.6))
        self._etiqueta(c, n, label, lado, dist=0.45 if not led else 0.9)

    def tierra(self, p):
        x, y = p
        self.linea((x, y), (x, y - 0.3))
        for i, w in enumerate((0.36, 0.24, 0.12)):
            yy = y - 0.3 - i * 0.11
            self.linea((x - w, yy), (x + w, yy), lw=1.8)

    def terminal(self, p, t, lado='izq', size=13):
        self.ax.add_patch(Circle(p, 0.1, fc='white', ec=INK, lw=1.6, zorder=5))
        dx = -0.2 if lado == 'izq' else 0.2
        self.texto((p[0] + dx, p[1]), t, ha='right' if lado == 'izq' else 'left', size=size)

    def fuente(self, p_abajo, p_arriba, label=None, tipo='ac'):
        p1, p2, L, u, n = self._eje(p_abajo, p_arriba)
        c = (p1 + p2) / 2
        r = 0.42
        self.linea(tuple(p1), tuple(c - u * r))
        self.linea(tuple(c + u * r), tuple(p2))
        self.ax.add_patch(Circle(c, r, fill=False, color=INK, lw=self.lw))
        if tipo == 'ac':
            t = np.linspace(-1, 1, 40)
            self.ax.plot(c[0] + t * 0.25, c[1] + 0.12 * np.sin(np.pi * t), color=INK, lw=1.6)
        else:
            self.texto(tuple(c + u * 0.17), '+', size=11)
            self.texto(tuple(c - u * 0.17), '−', size=11)
        if label:
            self.texto((c[0] + 0.6, c[1]), label, ha='left')

    def opamp(self, x, y, ancho=1.8, alto=1.8, mas_arriba=False, label=None):
        """x, y: centro del lado de las entradas. Devuelve pines: 'inv', 'noinv', 'out'."""
        tri = [(x, y + alto / 2), (x, y - alto / 2), (x + ancho, y)]
        self.ax.add_patch(Polygon(tri, closed=True, fc='#F4F6F7', ec=INK, lw=self.lw, zorder=3))
        dy = alto * 0.25
        top, bot = (x, y + dy), (x, y - dy)
        inv, noinv = (bot, top) if mas_arriba else (top, bot)
        self.texto((x + 0.25, inv[1]), '−', size=15, weight='bold')
        self.texto((x + 0.25, noinv[1]), '+', size=15, weight='bold')
        if label:
            self.texto((x + ancho * 0.42, y), label, size=10, color='#4F606B')
        return {'inv': inv, 'noinv': noinv, 'out': (x + ancho, y)}

    def interruptor(self, p1, p2, label=None, lado=1):
        p1, p2, L, u, n = self._eje(p1, p2)
        c = (p1 + p2) / 2
        a = c - u * 0.35
        b = c + u * 0.35
        self.linea(tuple(p1), tuple(a))
        self.linea(tuple(b), tuple(p2))
        self.ax.add_patch(Circle(a, 0.07, fc='white', ec=INK, zorder=5))
        self.ax.add_patch(Circle(b, 0.07, fc='white', ec=INK, zorder=5))
        self.linea(tuple(a + n * 0.32), tuple(b + n * 0.32), lw=2)
        self.linea(tuple(c + n * 0.32), tuple(c + n * 0.55), lw=2)
        self._etiqueta(c, n, label, lado, dist=0.8)

    def parlante(self, p, label=None):
        x, y = p
        self.ax.add_patch(Polygon([(x, y - 0.25), (x, y + 0.25), (x + 0.3, y + 0.25), (x + 0.3, y - 0.25)], closed=True, fc='white', ec=INK, lw=1.8))
        self.ax.add_patch(Polygon([(x + 0.3, y - 0.25), (x + 0.3, y + 0.25), (x + 0.7, y + 0.6), (x + 0.7, y - 0.6)], closed=True, fc='white', ec=INK, lw=1.8))
        if label:
            self.texto((x + 0.35, y - 0.9), label, size=12)

    def caja(self, x, y, w, h, t, color='#EAF1F4', borde=INK, size=13):
        self.ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.18', fc=color, ec=borde, lw=1.6, zorder=2))
        self.texto((x + w / 2, y + h / 2), t, size=size)

    def flecha(self, p1, p2, color=INK):
        self.ax.annotate('', xy=p2, xytext=p1, arrowprops=dict(arrowstyle='-|>', color=color, lw=1.8, shrinkA=0, shrinkB=0))

    def guardar(self, ruta, dpi=220):
        self.fig.savefig(ruta, dpi=dpi, facecolor=self.fig.get_facecolor())
        plt.close(self.fig)

    def mostrar(self):
        plt.show()


def ecuacion(latex, ruta=None, size=26, color=INK, dpi=300):
    """Renderiza una ecuación (sintaxis mathtext, parecida a LaTeX) como imagen PNG con fondo transparente."""
    fig = plt.figure(figsize=(0.01, 0.01))
    fig.text(0, 0, '$' + latex + '$', fontsize=size, color=color)
    if ruta:
        fig.savefig(ruta, dpi=dpi, bbox_inches='tight', pad_inches=0.04, transparent=True)
        plt.close(fig)
    return fig

## 0. Un mismo chip, tres comportamientos
- **Lazo abierto:** $v_O = A_d\,(v_+ - v_-)$ con $A_d \approx 10^5$ → la salida se satura en $V_{OH}$ o $V_{OL}$: **compara**.
- **Realimentación negativa:** la salida corrige la entrada y aparece el corto virtual $v_- \approx v_+$: **opera** (suma, resta, integra).
- **Realimentación positiva:** la salida refuerza el cambio y aparecen dos umbrales $V_1$ y $V_2$: **decide y recuerda** (Schmitt, osciladores).

In [ ]:
for nombre, modo in (('modo_abierto', 0), ('modo_negativa', 1), ('modo_positiva', 2)):
    s = Esquema(7.4, 4.6)
    A = s.opamp(3.0, 2.4, mas_arriba=(modo != 1))
    if modo == 0:
        s.terminal((1.0, A['noinv'][1]), r'$v_I$', size=12); s.linea((1.0, A['noinv'][1]), A['noinv'])
        s.terminal((1.0, A['inv'][1]), r'$V_{ref}$', size=12); s.linea((1.0, A['inv'][1]), A['inv'])
    elif modo == 1:
        s.terminal((0.7, A['inv'][1]), r'$v_I$', size=12); s.resistencia((0.7, A['inv'][1]), (2.4, A['inv'][1]), None); s.linea((2.4, A['inv'][1]), A['inv']); s.nodo((2.6, A['inv'][1]))
        s.linea((2.6, A['inv'][1]), (2.6, 4.0)); s.resistencia((2.6, 4.0), (5.4, 4.0), None, color='#2F6DB5'); s.linea((5.4, 4.0), (5.4, 2.4))
        s.linea(A['noinv'], (2.8, A['noinv'][1])); s.tierra((2.8, A['noinv'][1]))
    else:
        s.terminal((0.7, A['noinv'][1]), r'$v_I$', size=12); s.resistencia((0.7, A['noinv'][1]), (2.4, A['noinv'][1]), None); s.linea((2.4, A['noinv'][1]), A['noinv']); s.nodo((2.6, A['noinv'][1]))
        s.linea((2.6, A['noinv'][1]), (2.6, 4.0)); s.resistencia((2.6, 4.0), (5.4, 4.0), None, color=LED_COLORS['rojo']); s.linea((5.4, 4.0), (5.4, 2.4))
        s.linea(A['inv'], (2.8, A['inv'][1])); s.tierra((2.8, A['inv'][1]))
    s.linea(A['out'], (6.6, 2.4)); s.texto((6.7, 2.75), r'$v_O$', ha='left', size=12)
    if modo: s.nodo((5.4, 2.4))
    s.mostrar()

## 1. Lazo abierto: comparador con referencia
La referencia sale de un divisor desde +12 V:

$$V_{ref} = 12\,\mathrm{V}\cdot\dfrac{R_2}{R_1+R_2} = 12\,\mathrm{V}\cdot\dfrac{4{,}7\,\mathrm{k\Omega}}{10\,\mathrm{k\Omega}+4{,}7\,\mathrm{k\Omega}} = 3{,}84\,\mathrm{V}$$

$$v_O = \begin{cases} V_{OH} & \text{si } v_I > V_{ref}\\ V_{OL} & \text{si } v_I < V_{ref}\end{cases}\qquad t_{conm} \approx \dfrac{V_{OH}-V_{OL}}{SR} = \dfrac{21\,\mathrm{V}}{13\,\mathrm{V/\mu s}} \approx 1{,}6\,\mu\mathrm{s}$$

En montaje real conviene un 1N4148 en serie con el LED: en $V_{OL}$ el LED vería −10,5 V en inverso (su máximo es unos 5 V).

In [ ]:
s = Esquema(12.4, 6.4)
a = s.opamp(5.8, 3.2, mas_arriba=True, label='TL084')
s.terminal((1.9, a['noinv'][1]), r'$v_I$')
s.linea((1.9, a['noinv'][1]), a['noinv'])
s.terminal((1.9, a['inv'][1]), '+12 V')
s.resistencia((1.9, a['inv'][1]), (4.1, a['inv'][1]), r'$R_1$ 10 kΩ', lado=-1)
s.linea((4.1, a['inv'][1]), a['inv'])
s.nodo((4.8, a['inv'][1]))
s.resistencia((4.8, a['inv'][1]), (4.8, 0.9), r'$R_2$ 4,7 kΩ', lado=1)
s.tierra((4.8, 0.9))
s.texto((5.0, a['inv'][1] + 0.3), r'$V_{ref}$', ha='left', size=12, color='#2F6DB5')
s.linea(a['out'], (10.1, a['out'][1]))
s.nodo((9.2, a['out'][1]))
s.texto((9.2, a['out'][1] + 0.35), r'$v_O$', size=13)
s.resistencia((10.1, a['out'][1]), (10.1, 1.95), '820 Ω')
s.diodo((10.1, 1.95), (10.1, 0.9), 'LED', led='rojo', lado=-1)
s.tierra((10.1, 0.9))
s.mostrar()

### Comparador frente a Schmitt con una señal ruidosa
Sin histéresis, cada cruce del ruido por $V_{ref}$ produce una conmutación. Con realimentación positiva la ventana $V_H$ absorbe el ruido:

$$V_H = (V_{OH}-V_{OL})\,\dfrac{R_1}{R_1+R_F}$$

In [ ]:
def comparar(ruido=0.3, RF_k=100.0):
    rng = np.random.default_rng(7)
    t = np.linspace(0, 1, 3000)
    n = np.convolve(rng.normal(0, 1, t.size), np.ones(12) / 12, mode='same') * 3.5
    vI = 3.84 + 2.2 * np.sin(2 * np.pi * (t - 0.25)) + ruido * n
    Vref, R1, RF = 3.84, 10e3, RF_k * 1e3
    VH = (VOH - VOL) * R1 / (R1 + RF)
    comp = np.where(vI > Vref, VOH, VOL)
    sch = np.empty_like(vI); estado = VOL
    for i, v in enumerate(vI):
        if estado < 0 and v > Vref + VH / 2: estado = VOH
        elif estado > 0 and v < Vref - VH / 2: estado = VOL
        sch[i] = estado
    k1, k2 = np.count_nonzero(np.diff(comp)), np.count_nonzero(np.diff(sch))
    fig, ax = plt.subplots(3, 1, figsize=(10, 6.4), sharex=True)
    ax[0].plot(t, vI, color='#2F6DB5', lw=1); ax[0].axhline(Vref, color='#F2A33A', ls='--')
    ax[0].axhspan(Vref - VH / 2, Vref + VH / 2, color='#F2A33A', alpha=.18); ax[0].set_ylabel('$v_I$ (V)')
    ax[1].plot(t, comp, color='#D2402F'); ax[1].set_ylabel('comparador'); ax[1].set_title(f'{k1} conmutaciones', fontsize=10, loc='right')
    ax[2].plot(t, sch, color='#2E9B57'); ax[2].set_ylabel('Schmitt'); ax[2].set_title(f'{k2} conmutaciones (V_H = {coma(VH)} V)', fontsize=10, loc='right')
    ax[2].set_xlabel('tiempo (unidades arbitrarias)')
    for a in ax: a.grid(alpha=.25); a.spines[['top', 'right']].set_visible(False)
    plt.tight_layout(); plt.show()

interactivo(comparar, ruido=deslizador(0.3, 0, 1, 0.02, 'ruido (V)'), RF_k=deslizador(100, 20, 500, 5, 'R_F (kΩ)'))

## 2. Proyecto: semáforo hecho solo con operacionales
**Requisitos:** ciclo automático verde → amarillo → rojo (≈ 6 s, 2 s y 4 s), solo TL084 a ±12 V, LEDs a ≈ 10 mA.

**Arquitectura:** generador de rampa (A1 integrador + A2 Schmitt) → escalera de umbrales $V_A$, $V_B$ → comparadores C1, C2, C3 en lazo abierto → LEDs.
La idea clave es convertir tiempo en voltaje: $t = \Delta V / (dv/dt)$.

### Paso 1 · Umbrales del Schmitt (no inversor con referencia $V_r$)
$$V_H = (V_{OH}-V_{OL})\,\dfrac{R_1}{R_F} = 21\,\mathrm{V}\cdot\dfrac{10\,\mathrm{k\Omega}}{27\,\mathrm{k\Omega}} = 7{,}78\,\mathrm{V}\qquad V_r = 12\,\mathrm{V}\cdot\dfrac{10{,}5\,\mathrm{k\Omega}}{24\,\mathrm{k\Omega}+10{,}5\,\mathrm{k\Omega}} = 3{,}65\,\mathrm{V}$$

$$V_1 = V_r\left(1+\dfrac{R_1}{R_F}\right) - V_{OL}\,\dfrac{R_1}{R_F} = 8{,}89\,\mathrm{V}\qquad V_2 = V_r\left(1+\dfrac{R_1}{R_F}\right) - V_{OH}\,\dfrac{R_1}{R_F} = 1{,}12\,\mathrm{V}$$

Balance eléctrico: $R_+ = R_1\parallel R_F = 7{,}30\,\mathrm{k\Omega}$ y $R_- = 24\,\mathrm{k\Omega}\parallel 10{,}5\,\mathrm{k\Omega} = 7{,}30\,\mathrm{k\Omega}$.

### Paso 2 · Tiempos con $R_u$, $R_d$ y $C$
$$\dfrac{dv_{O1}}{dt} = \dfrac{|V_{OL}|-V_D}{R_u\,C} = \dfrac{9{,}9\,\mathrm{V}}{1{,}5\,\mathrm{M\Omega}\cdot 10\,\mu\mathrm{F}} = 0{,}66\,\mathrm{V/s}\qquad T_{sube} = \dfrac{V_H\,R_u\,C}{|V_{OL}|-V_D} = 11{,}8\,\mathrm{s}\qquad T_{baja} = \dfrac{V_H\,R_d\,C}{V_{OH}-V_D} = 94\,\mathrm{ms}$$

### Paso 3 · Escalera de umbrales
$$I = \dfrac{12\,\mathrm{V}}{R_a+R_b+R_c} = 1\,\mathrm{mA}\qquad V_A = I\,R_c = 5{,}1\,\mathrm{V}\qquad V_B = I\,(R_b+R_c) = 6{,}4\,\mathrm{V}$$

$$t_{verde} = \dfrac{V_A - V_2}{0{,}66\,\mathrm{V/s}} = 6{,}0\,\mathrm{s}\qquad t_{amarillo} = \dfrac{V_B - V_A}{0{,}66\,\mathrm{V/s}} = 1{,}97\,\mathrm{s}\qquad t_{rojo} = \dfrac{V_1 - V_B}{0{,}66\,\mathrm{V/s}} = 3{,}78\,\mathrm{s}$$

### Paso 4 · LEDs
$$R = \dfrac{V_{OH}-V_D-V_{LED}}{I_{LED}} = \dfrac{(10{,}5-0{,}6-2{,}0)\,\mathrm{V}}{9{,}6\,\mathrm{mA}} = 820\,\Omega\qquad R_{amarillo} = \dfrac{(21-2{,}1)\,\mathrm{V}}{10{,}5\,\mathrm{mA}} = 1{,}8\,\mathrm{k\Omega}$$

| Rampa | C1 | C2 | C3 | C2 − C1 | LED |
|---|---|---|---|---|---|
| $v < V_A$ | VOL | VOL | VOH | 0 V | **verde** |
| $V_A < v < V_B$ | VOL | VOH | VOL | +21 V | **amarillo** |
| $v > V_B$ | VOH | VOH | VOL | 0 V | **rojo** |

In [ ]:
# Verificación numérica del diseño del semáforo
R1, RF = 10e3, 27e3
Vr = 12 * 10.5e3 / (24e3 + 10.5e3)
VH = (VOH - VOL) * R1 / RF
V1 = Vr * (1 + R1 / RF) - VOL * R1 / RF
V2 = Vr * (1 + R1 / RF) - VOH * R1 / RF
Ru, Rd, C = 1.5e6, 12e3, 10e-6
pend = (-VOL - VD) / (Ru * C)
Ra, Rb, Rc = 5.6e3, 1.3e3, 5.1e3
I = 12 / (Ra + Rb + Rc); VA = I * Rc; VB = I * (Rb + Rc)
filas = [('V_r', Vr, 'V'), ('V_H', VH, 'V'), ('V_1', V1, 'V'), ('V_2', V2, 'V'),
         ('R+ = R1 || RF', paralelo(R1, RF) / 1e3, 'kΩ'), ('R- = 24k || 10,5k', paralelo(24e3, 10.5e3) / 1e3, 'kΩ'),
         ('pendiente', pend, 'V/s'), ('T sube', VH / pend, 's'), ('T baja', VH * Rd * C / (VOH - VD) * 1e3, 'ms'),
         ('V_A', VA, 'V'), ('V_B', VB, 'V'),
         ('t verde', (VA - V2) / pend, 's'), ('t amarillo', (VB - VA) / pend, 's'), ('t rojo', (V1 - VB) / pend, 's'),
         ('R LED verde/rojo', (VOH - VD - 2.0) / 9.6e-3, 'Ω'), ('R LED amarillo', (VOH - VOL - 2.1) / 10.5e-3, 'Ω')]
for n, v, u in filas:
    print(f'{n:>20s} = {coma(v, 3):>9s} {u}')

### Esquemas: generador de rampa y sección de comparadores

In [ ]:
s = Esquema(16.5, 8.2)
A1 = s.opamp(6, 4, label='A1')
s.linea(A1['inv'], (5.4, A1['inv'][1]))
s.linea((5.4, A1['inv'][1]), (5.4, 6.6))
s.condensador((5.4, 6.6), (8.4, 6.6), r'$C$ 10 µF')
s.linea((8.4, 6.6), (8.4, 4.0))
s.linea(A1['out'], (8.4, 4.0))
s.linea(A1['noinv'], (5.6, A1['noinv'][1]))
s.tierra((5.6, A1['noinv'][1]))
# red de diodos
s.linea((1.4, 4.45), (1.4, 5.5))
s.linea((1.4, 5.5), (1.9, 5.5)); s.linea((1.4, 4.45), (1.9, 4.45))
s.diodo((3.0, 5.5), (1.9, 5.5), r'$D_1$', lado=1)
s.resistencia((3.0, 5.5), (5.0, 5.5), r'$R_u$ 1,5 MΩ')
s.linea((5.0, 5.5), (5.4, 5.5)); s.nodo((5.4, 5.5))
s.diodo((1.9, 4.45), (3.0, 4.45), r'$D_2$', lado=-1)
s.resistencia((3.0, 4.45), (5.0, 4.45), r'$R_d$ 12 kΩ', lado=-1)
s.linea((5.0, 4.45), (5.4, 4.45)); s.nodo((5.4, 4.45))
s.nodo((1.4, 4.45)); s.nodo((1.4, 5.5))
# Schmitt
A2 = s.opamp(11.2, 4, mas_arriba=True, label='A2')
s.nodo((8.4, 4.45))
s.resistencia((8.4, 4.45), (10.4, 4.45), r'$R_1$ 10 kΩ', lado=-1)
s.linea((10.4, 4.45), A2['noinv']); s.nodo((10.6, 4.45))
s.linea((10.6, 4.45), (10.6, 5.7))
s.resistencia((10.6, 5.7), (13.6, 5.7), r'$R_F$ 27 kΩ')
s.linea((13.6, 5.7), (13.6, 4.0))
s.linea(A2['out'], (15.2, 4.0)); s.nodo((13.6, 4.0))
s.linea(A2['inv'], (10.3, A2['inv'][1]))
s.terminal((10.3, A2['inv'][1]), r'$V_r$ = 3,65 V', size=12)
# realimentación
s.linea((15.2, 4.0), (15.2, 0.9), (1.4, 0.9), (1.4, 4.45))
s.texto((15.35, 4.35), r'$v_{O2}$', ha='left')
s.texto((8.6, 2.95), r'$v_{O1}$ rampa', ha='left', color='#2F6DB5')
s.linea((8.4, 4.0), (8.4, 2.4)); s.terminal((8.4, 2.4), 'a comparadores', lado='der', size=11)
s.texto((7.0, 1.25), 'Rampa lenta por $R_u$ (salida en $V_{OL}$)  ·  retorno rápido por $R_d$ (salida en $V_{OH}$)', size=11, color='#4F606B')
s.mostrar()

In [ ]:
s = Esquema(15.6, 9.8)
Cr = s.opamp(6, 7.1, mas_arriba=True, label='C1')
Cy = s.opamp(6, 4.6, mas_arriba=True, label='C2')
Cg = s.opamp(6, 2.1, mas_arriba=False, label='C3')
# escalera
s.terminal((1.4, 9.3), '+12 V', lado='izq', size=12)
s.resistencia((1.4, 9.3), (1.4, Cr['inv'][1]), r'$R_a$ 5,6 kΩ', lado=-1)
s.nodo((1.4, Cr['inv'][1]))
s.linea_saltos((1.4, Cr['inv'][1]), Cr['inv'], saltos=[3.5])
s.resistencia((1.4, Cr['inv'][1]), (1.4, Cy['inv'][1]), r'$R_b$ 1,3 kΩ', lado=-1)
s.nodo((1.4, Cy['inv'][1]))
s.linea_saltos((1.4, Cy['inv'][1]), Cy['inv'], saltos=[3.5])
s.resistencia((1.4, Cy['inv'][1]), (1.4, 0.9), r'$R_c$ 5,1 kΩ', lado=-1)
s.tierra((1.4, 0.9))
s.nodo((2.5, Cy['inv'][1]))
s.linea((2.5, Cy['inv'][1]), (2.5, Cg['noinv'][1]), Cg['noinv'])
s.texto((4.6, Cr['inv'][1] - 0.3), r'$V_B$ = 6,4 V', size=11, color='#2F6DB5')
s.texto((4.6, Cy['inv'][1] - 0.3), r'$V_A$ = 5,1 V', size=11, color='#2F6DB5')
# bus de rampa
s.terminal((3.5, 9.3), 'rampa', lado='der', size=12)
s.linea((3.5, 9.3), (3.5, Cg['inv'][1]))
for pin in (Cr['noinv'], Cy['noinv'], Cg['inv']):
    s.linea((3.5, pin[1]), pin); s.nodo((3.5, pin[1]))
# rojo
s.linea(Cr['out'], (8.4, Cr['out'][1])); s.nodo((8.4, Cr['out'][1]))
s.diodo((8.4, Cr['out'][1]), (9.6, Cr['out'][1]), '1N4148')
s.resistencia((9.6, Cr['out'][1]), (11.4, Cr['out'][1]), '820 Ω')
s.diodo((11.4, Cr['out'][1]), (12.8, Cr['out'][1]), 'rojo', led='rojo')
s.linea((12.8, Cr['out'][1]), (13.6, Cr['out'][1])); s.tierra((13.6, Cr['out'][1]))
# amarillo entre C2 y C1
s.linea(Cy['out'], (8.4, Cy['out'][1]))
s.resistencia((8.4, Cy['out'][1]), (10.2, Cy['out'][1]), '1,8 kΩ', lado=-1)
s.diodo((10.2, Cy['out'][1]), (11.6, Cy['out'][1]), 'amarillo', led='amarillo', lado=-1)
s.linea((11.6, Cy['out'][1]), (12.2, Cy['out'][1]), (12.2, 5.8), (8.4, 5.8), (8.4, Cr['out'][1]))
# verde
s.linea(Cg['out'], (8.4, Cg['out'][1]))
s.diodo((8.4, Cg['out'][1]), (9.6, Cg['out'][1]), '1N4148')
s.resistencia((9.6, Cg['out'][1]), (11.4, Cg['out'][1]), '820 Ω')
s.diodo((11.4, Cg['out'][1]), (12.8, Cg['out'][1]), 'verde', led='verde')
s.linea((12.8, Cg['out'][1]), (13.6, Cg['out'][1])); s.tierra((13.6, Cg['out'][1]))
s.mostrar()

### Diagrama de flujo del funcionamiento

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 9.2))
ax.set_xlim(0, 10); ax.set_ylim(0, 11.2); ax.axis('off')
from matplotlib.patches import FancyBboxPatch, Polygon
def caja(x, y, w, h, t, fc='#EAF1F4', ec=INK, tc=INK):
    ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle='round,pad=0.02,rounding_size=0.25', fc=fc, ec=ec, lw=1.6))
    ax.text(x, y, t, ha='center', va='center', fontsize=12, color=tc)
def rombo(x, y, w, h, t):
    ax.add_patch(Polygon([(x, y + h / 2), (x + w / 2, y), (x, y - h / 2), (x - w / 2, y)], closed=True, fc='#FFF4DC', ec=INK, lw=1.6))
    ax.text(x, y, t, ha='center', va='center', fontsize=12)
def fl(p1, p2, t=None, tp=None):
    ax.annotate('', xy=p2, xytext=p1, arrowprops=dict(arrowstyle='-|>', color=INK, lw=1.5, shrinkA=0, shrinkB=0))
    if t: ax.text(*tp, t, fontsize=11, color='#4F606B', ha='center', va='center')
caja(3.4, 10.5, 3.6, 0.8, 'Encender: fuente ±12 V', fc='#12343F', tc='white')
caja(3.4, 9.15, 4.6, 0.9, 'A1 integra: la rampa sube\ncon pendiente 0,66 V/s')
rombo(3.4, 7.55, 3.8, 1.5, r'$v_{rampa} < V_A$ ?')
caja(8.2, 7.55, 2.6, 0.9, 'C3 → VERDE', fc='#D7F0E1')
rombo(3.4, 5.55, 3.8, 1.5, r'$v_{rampa} < V_B$ ?')
caja(8.2, 5.55, 2.6, 0.9, 'C2 − C1 →\nAMARILLO', fc='#FCEFC7')
rombo(3.4, 3.55, 3.8, 1.5, r'$v_{rampa} \geq V_1$ ?')
caja(8.2, 3.55, 2.6, 0.9, 'C1 → ROJO', fc='#F7D9D4')
caja(3.4, 1.35, 5.4, 1.1, 'A2 (Schmitt) conmuta a $V_{OH}$:\nretorno rápido por $R_d$ hasta $V_2$ (94 ms)')
fl((3.4, 10.1), (3.4, 9.6)); fl((3.4, 8.7), (3.4, 8.3))
fl((5.3, 7.55), (6.9, 7.55), 'sí', (6.1, 7.85)); fl((3.4, 6.8), (3.4, 6.3), 'no', (3.75, 6.55))
fl((5.3, 5.55), (6.9, 5.55), 'sí', (6.1, 5.85)); fl((3.4, 4.8), (3.4, 4.3), 'no', (3.75, 4.55))
fl((5.3, 3.55), (6.9, 3.55), 'no', (6.1, 3.85)); fl((3.4, 2.8), (3.4, 1.9), 'sí', (3.75, 2.35))
for y in (7.55, 5.55, 3.55):
    ax.plot([9.5, 9.75, 9.75], [y, y, 9.15], color=INK, lw=1.5)
ax.annotate('', xy=(5.7, 9.15), xytext=(9.75, 9.15), arrowprops=dict(arrowstyle='-|>', color=INK, lw=1.5))
ax.plot([0.7, 0.45, 0.45], [1.35, 1.35, 9.15], color=INK, lw=1.5)
ax.annotate('', xy=(1.1, 9.15), xytext=(0.45, 9.15), arrowprops=dict(arrowstyle='-|>', color=INK, lw=1.5))
ax.text(9.9, 6.4, 'la rampa sigue subiendo', rotation=90, fontsize=10, color='#4F606B', va='center')
plt.show()

### Simulación de la rampa y de los LEDs

In [ ]:
VOH, VOL, VD = 10.5, -10.5, 0.6
R1, RF, Vr = 10e3, 27e3, 12 * 10.5 / 34.5
V1 = Vr * (1 + R1 / RF) - VOL * R1 / RF
V2 = Vr * (1 + R1 / RF) - VOH * R1 / RF
Ru, Rd, C = 1.5e6, 12e3, 10e-6
VA, VB = 12 * 5.1 / 12, 12 * 6.4 / 12
dt = 1e-3; t = np.arange(0, 30, dt); v = np.zeros_like(t); vs = np.zeros_like(t)
x, out = V2, VOL
for i in range(len(t)):
    if out < 0: x += (-VOL - VD) / (Ru * C) * dt
    else: x -= (VOH - VD) / (Rd * C) * dt
    vp = x * RF / (R1 + RF) + out * R1 / (R1 + RF)
    if out < 0 and vp > Vr: out = VOH
    elif out > 0 and vp < Vr: out = VOL
    v[i] = x; vs[i] = out
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 5.2), sharex=True, gridspec_kw={'height_ratios': [3, 1.2]})
a1.plot(t, v, color='#2F6DB5', lw=2.2, label='rampa $v_{O1}$')
for y, c, l in ((VA, LED_COLORS['verde'], '$V_A$ = 5,1 V'), (VB, LED_COLORS['rojo'], '$V_B$ = 6,4 V')):
    a1.axhline(y, color=c, ls='--', lw=1.3); a1.text(30.2, y, l, va='center', fontsize=11, color=c)
for y, l in ((V1, '$V_1$ = 8,89 V'), (V2, '$V_2$ = 1,12 V')):
    a1.axhline(y, color='#8FA3B0', ls=':', lw=1.2); a1.text(30.2, y, l, va='center', fontsize=11, color='#4F606B')
a1.set_ylabel('voltaje (V)'); a1.set_ylim(0, 10); a1.grid(alpha=.25); a1.spines[['top', 'right']].set_visible(False)
g = v < VA; yl = (v >= VA) & (v < VB); r = v >= VB
for k, (m, c) in enumerate(((r, 'rojo'), (yl, 'amarillo'), (g, 'verde'))):
    a2.fill_between(t, k, k + 0.8, where=m, color=LED_COLORS[c], step='mid')
a2.set_yticks([0.4, 1.4, 2.4]); a2.set_yticklabels(['rojo', 'amarillo', 'verde'])
a2.set_xlabel('tiempo (s)'); a2.set_xlim(0, 30); a2.spines[['top', 'right']].set_visible(False)
fig.tight_layout(); plt.show()

### Semáforo interactivo: cambie $R_u$, $C$, $V_A$ y $V_B$

In [ ]:
def semaforo(Ru_MOhm=1.5, C_uF=10.0, VA=5.1, VB=6.4):
    R1, RF, Rd = 10e3, 27e3, 12e3
    Vr = 12 * 10.5 / 34.5
    V1 = Vr * (1 + R1 / RF) - VOL * R1 / RF; V2 = Vr * (1 + R1 / RF) - VOH * R1 / RF
    Ru, C = Ru_MOhm * 1e6, C_uF * 1e-6
    dt = 2e-3; t = np.arange(0, 40, dt); v = np.empty_like(t); x, out = V2, VOL
    for i in range(t.size):
        x += ((-VOL - VD) / (Ru * C) if out < 0 else -(VOH - VD) / (Rd * C)) * dt
        vp = x * RF / (R1 + RF) + out * R1 / (R1 + RF)
        if out < 0 and vp > Vr: out = VOH
        elif out > 0 and vp < Vr: out = VOL
        v[i] = x
    p = (-VOL - VD) / (Ru * C)
    tg, ty, tr = max(0, min(VA, V1) - V2) / p, max(0, min(VB, V1) - max(VA, V2)) / p, max(0, V1 - max(VB, V2)) / p
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 4.8), sharex=True, gridspec_kw={'height_ratios': [3, 1.1]})
    a1.plot(t, v, color='#2F6DB5', lw=2)
    for y, c in ((VA, '#2E9B57'), (VB, '#D2402F')): a1.axhline(y, color=c, ls='--', lw=1.2)
    a1.set_ylabel('$v_{O1}$ (V)'); a1.set_ylim(0, 10); a1.grid(alpha=.25)
    a1.set_title(f'verde {coma(tg)} s · amarillo {coma(ty)} s · rojo {coma(tr)} s · ciclo {coma(tg + ty + tr + (V1 - V2) * Rd * C / (VOH - VD))} s', fontsize=11)
    for k, (m, c) in enumerate(((v >= VB, '#D2402F'), ((v >= VA) & (v < VB), '#E8A317'), (v < VA, '#2E9B57'))):
        a2.fill_between(t, k, k + 0.8, where=m, color=c, step='mid')
    a2.set_yticks([0.4, 1.4, 2.4]); a2.set_yticklabels(['rojo', 'amarillo', 'verde']); a2.set_xlabel('tiempo (s)')
    for a in (a1, a2): a.spines[['top', 'right']].set_visible(False)
    plt.tight_layout(); plt.show()

interactivo(semaforo, Ru_MOhm=deslizador(1.5, 0.5, 3, 0.1, 'R_u (MΩ)'), C_uF=deslizador(10, 4.7, 22, 0.1, 'C (µF)'),
            VA=deslizador(5.1, 2, 8, 0.05, 'V_A (V)'), VB=deslizador(6.4, 2.2, 8.8, 0.05, 'V_B (V)'))

## 3. Realimentación negativa
### 3.1 Sumador inversor: convertidor D/A de 3 bits
Con la tierra virtual, las corrientes de cada bit se suman en $R_F$:

$$v_{O1} = -R_F\left(\dfrac{v_{b2}}{10\,\mathrm{k\Omega}}+\dfrac{v_{b1}}{20\,\mathrm{k\Omega}}+\dfrac{v_{b0}}{40\,\mathrm{k\Omega}}\right)\qquad v_O = -v_{O1} = \dfrac{5\,\mathrm{V}\cdot 8\,\mathrm{k\Omega}}{40\,\mathrm{k\Omega}}\,(4b_2+2b_1+b_0) = 1\,\mathrm{V}\cdot N$$

In [ ]:
s = Esquema(16.2, 7.9)
A = s.opamp(8, 3.6, label='A1')
for i, (b, R, y) in enumerate((('$b_2$ (MSB)', '10 kΩ', 6.1), ('$b_1$', '20 kΩ', 5.05), ('$b_0$ (LSB)', '40 kΩ', 4.05))):
    s.terminal((1.6, y), b, size=12)
    s.resistencia((1.6, y), (5.0, y), R)
    s.linea((5.0, y), (6.6, y)); s.nodo((6.6, y))
s.linea((6.6, 6.1), (6.6, A['inv'][1]), A['inv'])
s.linea((6.6, 6.1), (6.6, 6.9)); s.resistencia((6.6, 6.9), (10.3, 6.9), r'$R_F$ 8 kΩ'); s.linea((10.3, 6.9), (10.3, 3.6))
s.linea(A['out'], (10.3, 3.6)); s.nodo((10.3, 3.6))
s.linea(A['noinv'], (7.6, A['noinv'][1])); s.tierra((7.6, A['noinv'][1]))
B = s.opamp(13.0, 3.15, label='A2')
s.resistencia((10.3, 3.6), (12.3, 3.6), '10 kΩ', lado=-1)
s.linea((12.3, 3.6), B['inv']); s.nodo((12.5, 3.6))
s.linea((12.5, 3.6), (12.5, 5.0)); s.resistencia((12.5, 5.0), (15.2, 5.0), '10 kΩ'); s.linea((15.2, 5.0), (15.2, 3.15))
s.linea(B['out'], (15.6, 3.15)); s.nodo((15.2, 3.15))
s.linea(B['noinv'], (12.7, B['noinv'][1])); s.tierra((12.7, B['noinv'][1]))
s.texto((10.45, 4.05), r'$-v_{O1}$', ha='left', size=11, color='#2F6DB5')
s.texto((15.6, 2.7), r'$v_O$', ha='center')
s.texto((4.0, 1.6), 'bits de 0 V o 5 V (salidas de un microcontrolador)', size=11, color='#4F606B')
s.mostrar()

In [ ]:
def dac(RF_k=8.0):
    N = np.arange(8)
    b = [(N >> k) & 1 for k in (2, 1, 0)]
    vO1 = -RF_k * 1e3 * (5 * b[0] / 10e3 + 5 * b[1] / 20e3 + 5 * b[2] / 40e3)
    vO = -np.maximum(vO1, VOL)
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    ax.bar(N, vO, color=['#1E7F86' if x < VOH else '#D2402F' for x in vO])
    ax.axhline(VOH, color='#D2402F', ls='--', lw=1); ax.set_xticks(N); ax.set_xticklabels([format(n, '03b') for n in N])
    ax.set_ylabel('$v_O$ (V)'); ax.set_title(f'paso = {coma(RF_k * 5 / 40, 3)} V por código', fontsize=11)
    ax.grid(alpha=.25, axis='y'); ax.spines[['top', 'right']].set_visible(False); plt.tight_layout(); plt.show()

interactivo(dac, RF_k=deslizador(8, 4, 12, 0.1, 'R_F (kΩ)'))

### 3.2 Restador diferencial: medición de corriente con shunt
$$v_O = \dfrac{R_2}{R_1}\,(v_2-v_1)\quad\text{si}\quad \dfrac{R_4}{R_3}=\dfrac{R_2}{R_1}\qquad v_O = 16{,}5\cdot R_S\,I = 1{,}65\,\dfrac{\mathrm{V}}{\mathrm{A}}\cdot I$$

$$CMRR \approx \dfrac{1+R_2/R_1}{4\,\varepsilon} = \dfrac{17{,}5}{4\cdot 0{,}01} = 438 \;\to\; 52{,}8\,\mathrm{dB}$$

In [ ]:
s = Esquema(15.6, 7.9)
A = s.opamp(9.6, 3.8, mas_arriba=True, label='A1')
s.terminal((1.6, 6.4), '+12 V', size=12)
s.linea((1.6, 6.4), (2.2, 6.4))
s.caja(2.2, 5.85, 2.0, 1.1, 'carga\n(motor)', color='#FCEFC7')
s.linea((4.2, 6.4), (5.0, 6.4), (5.0, 5.0)); s.nodo((5.0, 5.0))
s.resistencia((5.0, 5.0), (5.0, 2.2), r'$R_S$ 0,1 Ω', lado=-1); s.nodo((5.0, 2.2))
s.linea((5.0, 2.2), (5.0, 1.2)); s.tierra((5.0, 1.2))
s.flecha((5.45, 4.2), (5.45, 3.2), color=LED_COLORS['rojo']); s.texto((5.6, 3.7), r'$I$', ha='left', size=13, color=LED_COLORS['rojo'])
s.texto((4.7, 5.25), r'$v_2$', size=12, color='#2F6DB5'); s.texto((4.7, 1.95), r'$v_1$', size=12, color='#2F6DB5')
s.linea((5.0, 5.0), (6.0, 5.0), (6.0, 4.25))
s.resistencia((6.0, 4.25), (8.2, 4.25), r'$R_3$ 10 kΩ'); s.linea((8.2, 4.25), A['noinv']); s.nodo((8.8, 4.25))
s.resistencia((8.8, 4.25), (8.8, 0.6), r'$R_4$ 165 kΩ', lado=1); s.tierra((8.8, 0.6))
s.linea((5.0, 2.2), (6.0, 2.2), (6.0, 3.35))
s.resistencia((6.0, 3.35), (7.9, 3.35), r'$R_1$ 10 kΩ', lado=-1)
s.linea_saltos((7.9, 3.35), A['inv'], saltos=(8.8,)); s.nodo((9.35, 3.35))
s.linea((9.35, 3.35), (9.35, 6.6)); s.resistencia((9.35, 6.6), (12.4, 6.6), r'$R_2$ 165 kΩ'); s.linea((12.4, 6.6), (12.4, 3.8))
s.linea(A['out'], (13.2, 3.8)); s.nodo((12.4, 3.8)); s.texto((13.35, 3.8), r'$v_O$ → ADC', ha='left')
s.mostrar()

In [ ]:
# Monte Carlo: ¿cuánto rechazo de modo común se obtiene con resistencias de 1 %?
rng = np.random.default_rng(1)
n, tol = 20000, 0.01
R1 = 10e3 * (1 + rng.uniform(-tol, tol, n)); R3 = 10e3 * (1 + rng.uniform(-tol, tol, n))
R2 = 165e3 * (1 + rng.uniform(-tol, tol, n)); R4 = 165e3 * (1 + rng.uniform(-tol, tol, n))
Ad = 0.5 * (R4 / (R3 + R4) * (1 + R2 / R1) + R2 / R1)
Acm = np.abs(R4 / (R3 + R4) * (1 + R2 / R1) - R2 / R1)
cmrr = 20 * np.log10(Ad / Acm)
print(f'CMRR peor caso analítico ≈ {coma(20*np.log10(17.5/(4*tol)),1)} dB; mediana Monte Carlo = {coma(np.median(cmrr),1)} dB; percentil 1 % = {coma(np.percentile(cmrr,1),1)} dB')
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(np.clip(cmrr, 40, 110), bins=70, color='#1E7F86')
ax.axvline(20 * np.log10(17.5 / (4 * tol)), color='#D2402F', ls='--', label='peor caso')
ax.set_xlabel('CMRR (dB)'); ax.set_ylabel('casos'); ax.legend(frameon=False); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

### 3.3 Amplificador multientradas universal: $v_O = 2v_1 - 3v_2 + v_3$
Método de Gabriunas (balance matemático y eléctrico a la vez):
1. Cada coeficiente es $a_k = R_F/R_k$: con $R_F = 60\,\mathrm{k\Omega}$, $R_2 = 20\,\mathrm{k\Omega}$, $R_1 = 30\,\mathrm{k\Omega}$, $R_3 = 60\,\mathrm{k\Omega}$.
2. La ganancia positiva total es $A^+ = A^- + 1 = 4$; los coeficientes positivos suman 3, falta 1: $R_{aux+} = R_F/1 = 60\,\mathrm{k\Omega}$ a tierra.
3. Comprobación: $R_+ = 30\parallel 60\parallel 60 = 15\,\mathrm{k\Omega}$ y $R_- = 20\parallel 60 = 15\,\mathrm{k\Omega}$.

La función siguiente aplica el método a cualquier combinación de coeficientes y verifica el resultado resolviendo el circuito.

In [ ]:
s = Esquema(14.4, 7.6)
A = s.opamp(8.2, 3.8, mas_arriba=True, label='A1')
s.terminal((1.4, 6.2), r'$v_1$'); s.resistencia((1.4, 6.2), (4.6, 6.2), '30 kΩ')
s.terminal((1.4, 5.1), r'$v_3$'); s.resistencia((1.4, 5.1), (4.6, 5.1), '60 kΩ')
s.linea((4.6, 6.2), (6.4, 6.2), (6.4, A['noinv'][1])); s.linea((4.6, 5.1), (6.4, 5.1)); s.nodo((6.4, 5.1))
s.linea((6.4, A['noinv'][1]), A['noinv']); s.nodo((6.4, A['noinv'][1]))
s.resistencia((6.4, A['noinv'][1]), (6.4, 2.95), r'$R_{aux+}$ 60 kΩ', lado=-1)
s.linea((6.4, 2.95), (6.4, 2.6)); s.tierra((6.4, 2.6))
s.terminal((1.4, 1.6), r'$v_2$'); s.resistencia((1.4, 1.6), (4.6, 1.6), '20 kΩ')
s.linea((4.6, 1.6), (7.4, 1.6), (7.4, A['inv'][1]), A['inv']); s.nodo((7.4, 1.6))
s.linea((7.4, 1.6), (7.4, 0.9)); s.resistencia((7.4, 0.9), (10.9, 0.9), r'$R_F$ 60 kΩ', lado=-1)
s.linea((10.9, 0.9), (10.9, 3.8)); s.linea(A['out'], (12.4, 3.8)); s.nodo((10.9, 3.8))
s.texto((12.6, 3.8), r'$v_O$', ha='left')
s.mostrar()

In [ ]:
def disenar_multientradas(pos, neg, RF=60e3):
    """pos, neg: diccionarios {nombre: coeficiente>0}. Devuelve las resistencias (método de Gabriunas)."""
    sp, sn = sum(pos.values()), sum(neg.values())
    aux_pos = max(0.0, sn + 1 - sp)   # completa A+ = A- + 1
    aux_neg = max(0.0, sp - 1 - sn)   # si sobra ganancia positiva, se agrega en el lado negativo
    R = {f'R_{k} (+)': RF / a for k, a in pos.items()}
    R.update({f'R_{k} (−)': RF / a for k, a in neg.items()})
    if aux_pos: R['R_aux+ (a tierra)'] = RF / aux_pos
    if aux_neg: R['R_aux− (a tierra)'] = RF / aux_neg
    Rp = 1 / (sum(pos.values()) / RF + aux_pos / RF)
    Rn = 1 / (sum(neg.values()) / RF + aux_neg / RF + 1 / RF)
    return R, Rp, Rn

def verificar(pos, neg, RF, R, v):
    """Resuelve el circuito ideal: v+ por el divisor del lado positivo y vO = v+(1 + RF/R-par) − RF·Σ v/R."""
    gp = {k: 1 / R[f'R_{k} (+)'] for k in pos}
    g_aux = 1 / R['R_aux+ (a tierra)'] if 'R_aux+ (a tierra)' in R else 0
    vp = sum(gp[k] * v[k] for k in gp) / (sum(gp.values()) + g_aux)
    gn = {k: 1 / R[f'R_{k} (−)'] for k in neg}
    gn_aux = 1 / R['R_aux− (a tierra)'] if 'R_aux− (a tierra)' in R else 0
    return vp * (1 + RF * (sum(gn.values()) + gn_aux)) - RF * sum(gn[k] * v[k] for k in gn)

pos, neg, RF = {'1': 2, '3': 1}, {'2': 3}, 60e3
R, Rp, Rn = disenar_multientradas(pos, neg, RF)
for k, r in R.items(): print(f'{k:>18s} = {coma(r / 1e3, 2)} kΩ')
print(f'Balance: R+ = {coma(Rp / 1e3, 2)} kΩ, R− = {coma(Rn / 1e3, 2)} kΩ')
v = {'1': 1.0, '2': 0.5, '3': 2.0}
print(f'Prueba v1=1, v2=0,5, v3=2 → vO = {coma(verificar(pos, neg, RF, R, v), 3)} V (esperado {coma(2*1 - 3*0.5 + 2, 3)} V)')

## 4. Realimentación positiva
### 4.1 Termostato con Schmitt inversor
LM35 (10 mV/°C) × 10 → $v_T$ = 100 mV/°C. Calefactor encendido por debajo de 29 °C y apagado por encima de 31 °C: umbrales de 2,9 V y 3,1 V, $V_H = 0{,}198\,\mathrm{V}$.

$$V_1 = \dfrac{V\,R_F + V_{OH}\,R_1}{R_F+R_1}\qquad V_2 = \dfrac{V\,R_F + V_{OL}\,R_1}{R_F+R_1}\qquad \dfrac{V_{OH}-V_{OL}}{V_H} = 1+\dfrac{R_F}{R_1}\;\Rightarrow\;\dfrac{21}{0{,}198} = 106\;\Rightarrow\;R_F = 105\,\mathrm{k\Omega}$$

$$V = \dfrac{V_1(R_F+R_1) - V_{OH}R_1}{R_F} = \dfrac{3{,}1\cdot 106 - 10{,}5}{105} = 3{,}03\,\mathrm{V}\qquad R_A = R_1\dfrac{12\,\mathrm{V}}{V} = 3{,}96\,\mathrm{k\Omega}\;(3{,}92)\qquad R_B = R_1\dfrac{12\,\mathrm{V}}{12\,\mathrm{V}-V} = 1{,}34\,\mathrm{k\Omega}\;(1{,}33)$$

In [ ]:
s = Esquema(20.2, 7.4)
s.caja(0.3, 3.0, 1.9, 1.4, 'LM35\n10 mV/°C', color='#EAF1F4')
A = s.opamp(3.2, 3.7, mas_arriba=True, label='×10')
s.linea((2.2, 3.7), (2.7, 3.7), (2.7, A['noinv'][1]), A['noinv'])
s.linea(A['inv'], (2.9, A['inv'][1]), (2.9, 2.2)); s.nodo((2.9, 2.2))
s.resistencia((2.9, 2.2), (2.9, 0.9), '10 kΩ', lado=-1); s.tierra((2.9, 0.9))
s.resistencia((2.9, 2.2), (5.6, 2.2), '90 kΩ', lado=-1); s.linea((5.6, 2.2), (5.6, 3.7))
s.linea(A['out'], (6.2, 3.7)); s.nodo((5.6, 3.7))
s.texto((5.3, 4.5), r'$v_T$ = 100 mV/°C', ha='left', size=12, color='#2F6DB5')
B = s.opamp(10.6, 3.7, mas_arriba=True, label='Schmitt')
s.linea((6.2, 3.7), (6.2, B['inv'][1]))
s.linea_saltos((6.2, B['inv'][1]), B['inv'], saltos=(7.4,))
s.linea((7.4, 4.15), B['noinv']); s.nodo((8.4, 4.15)); s.nodo((9.4, 4.15))
s.terminal((8.4, 6.7), '+12 V', size=12)
s.resistencia((8.4, 6.7), (8.4, 4.15), r'$R_A$ 3,92 kΩ', lado=-1)
s.resistencia((7.4, 4.15), (7.4, 0.6), r'$R_B$ 1,33 kΩ', lado=1); s.tierra((7.4, 0.6))
s.linea((9.4, 4.15), (9.4, 5.6)); s.resistencia((9.4, 5.6), (13.0, 5.6), r'$R_F$ 105 kΩ'); s.linea((13.0, 5.6), (13.0, 3.7))
s.linea(B['out'], (13.6, 3.7)); s.nodo((13.0, 3.7))
s.texto((13.15, 3.35), r'$v_O$', ha='left', size=12)
s.caja(13.6, 3.0, 2.6, 1.4, 'driver\nNPN + relé', color='#FCEFC7')
s.linea((16.2, 3.7), (17.0, 3.7))
s.caja(17.0, 3.0, 2.6, 1.4, 'calefactor', color='#F7D9D4')
s.mostrar()

In [ ]:
# Diseño del termostato a partir de las temperaturas de conmutación
T_on, T_off, k = 29.0, 31.0, 0.1          # °C, °C, V/°C
V1, V2 = T_off * k, T_on * k
R1 = 1e3
RF = R1 * ((VOH - VOL) / (V1 - V2) - 1)
V = (V1 * (RF + R1) - VOH * R1) / RF
RA, RB = R1 * 12 / V, R1 * 12 / (12 - V)
print(f'V_H = {coma(V1 - V2, 3)} V   R_F = {coma(RF / 1e3, 1)} kΩ   V = {coma(V, 4)} V')
print(f'Valor comercial R_F = 105 kΩ → V_H = {coma(21 * R1 / (R1 + 105e3), 3)} V')
print(f'R_A = {coma(RA / 1e3, 3)} kΩ   R_B = {coma(RB / 1e3, 3)} kΩ   (comerciales 3,92 kΩ y 1,33 kΩ; R_A || R_B = {coma(paralelo(3.92e3, 1.33e3) / 1e3, 3)} kΩ)')

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.plot([20, 31, 31], [10.5, 10.5, -10.5], color=LED_COLORS['rojo'], lw=2.4)
ax.plot([40, 29, 29], [-10.5, -10.5, 10.5], color='#2F6DB5', lw=2.4)
ax.annotate('', xy=(26, 10.5), xytext=(23, 10.5), arrowprops=dict(arrowstyle='-|>', color=LED_COLORS['rojo'], lw=2))
ax.annotate('', xy=(34, -10.5), xytext=(37, -10.5), arrowprops=dict(arrowstyle='-|>', color='#2F6DB5', lw=2))
ax.text(31.2, 0, 'apaga a 31 °C\n($v_T$ = 3,1 V)', fontsize=11, va='center')
ax.text(28.8, 0, 'enciende a 29 °C\n($v_T$ = 2,9 V)', fontsize=11, va='center', ha='right')
ax.set_xlim(20, 40); ax.set_ylim(-13, 13); ax.set_xlabel('temperatura (°C)'); ax.set_ylabel('$v_O$ (V)')
ax.grid(alpha=.25); ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout(); plt.show()

### ON/OFF con histéresis frente a control proporcional ($u = K_p\,(T_{set} - T)$)

In [ ]:
def planta(modo, T0=22.0, Tset=30.0, tf=1500, dt=0.5):
    Tamb, tau, dTmax = 22.0, 120.0, 30.0
    T = T0; on = 1; Ts = []; Us = []; Tm = T0
    for k in np.arange(0, tf, dt):
        Tm += (T - Tm) * dt / 15.0  # retardo del sensor
        if modo == 'onoff':
            if on and Tm > Tset + 1: on = 0
            elif not on and Tm < Tset - 1: on = 1
            u = on
        else:
            u = np.clip(0.5 * (Tset - Tm), 0, 1)
        T += dt / tau * (Tamb + dTmax * u - T)
        Ts.append(T); Us.append(u)
    return np.arange(0, tf, dt), np.array(Ts), np.array(Us)
tt, T1_, U1 = planta('onoff'); tt2, T2_, U2 = planta('p')
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.plot(tt, T1_, color=LED_COLORS['rojo'], lw=2, label='ON/OFF con Schmitt (±1 °C)')
ax.plot(tt2, T2_, color='#2F6DB5', lw=2, label='Proporcional, Kp = 0,5 /°C')
ax.axhspan(29, 31, color='#E8A317', alpha=.12); ax.axhline(30, color='#4F606B', ls='--', lw=1)
ax.text(1510, 30, 'consigna 30 °C', va='center', fontsize=11, color='#4F606B')
ax.set_xlabel('tiempo (s)'); ax.set_ylabel('temperatura (°C)'); ax.set_xlim(0, 1500); ax.grid(alpha=.25)
ax.legend(loc='lower right', frameon=False); ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout(); plt.show()

In [ ]:
def termostato(consigna=30.0, histeresis=2.0, Kp=0.5, ruido=0.1):
    rng = np.random.default_rng(4)
    Tamb, tau, dTmax, dt = 22.0, 120.0, 30.0, 0.5
    t = np.arange(0, 1500, dt)
    TA = TB = TmA = TmB = 22.0; on = 1; conm = 0; hA, hB = [], []
    for _ in t:
        TmA += (TA - TmA) * dt / 15; TmB += (TB - TmB) * dt / 15
        m = TmA + ruido * rng.normal()
        if on and m > consigna + histeresis / 2: on = 0; conm += 1
        elif not on and m < consigna - histeresis / 2: on = 1; conm += 1
        u = np.clip(Kp * (consigna - TmB - ruido * rng.normal()), 0, 1)
        TA += dt / tau * (Tamb + dTmax * on - TA); TB += dt / tau * (Tamb + dTmax * u - TB)
        hA.append(TA); hB.append(TB)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t, hA, color='#D2402F', label=f'ON/OFF ({conm} conmutaciones del relé)')
    ax.plot(t, hB, color='#2F6DB5', label=f'P, error final {coma(consigna - hB[-1], 2)} °C')
    ax.axhspan(consigna - histeresis / 2, consigna + histeresis / 2, color='#F2A33A', alpha=.15)
    ax.set_xlabel('tiempo (s)'); ax.set_ylabel('T (°C)'); ax.legend(frameon=False); ax.grid(alpha=.25)
    ax.spines[['top', 'right']].set_visible(False); plt.tight_layout(); plt.show()

interactivo(termostato, consigna=deslizador(30, 25, 45, 0.5, 'consigna (°C)'), histeresis=deslizador(2, 0, 6, 0.1, 'histéresis (°C)'),
            Kp=deslizador(0.5, 0.1, 3, 0.05, 'K_p (1/°C)'), ruido=deslizador(0.1, 0, 0.5, 0.01, 'ruido (°C)'))

### 4.2 Juego de reflejos: «detén la luz en la zona verde»
Oscilador Miller-Schmitt + comparador de ventana + restador + biestable:

$$f = \dfrac{R_F}{4\,R\,C\,R_1} = \dfrac{21\,\mathrm{k\Omega}}{4\cdot 510\,\mathrm{k\Omega}\cdot 1\,\mu\mathrm{F}\cdot 10\,\mathrm{k\Omega}} = 1{,}03\,\mathrm{Hz}\qquad V_{pico} = V_{OH}\dfrac{R_1}{R_F} = 5\,\mathrm{V}$$

$$t_{ventana} = \dfrac{2\cdot 0{,}5\,\mathrm{V}}{20{,}6\,\mathrm{V/s}} = 48\,\mathrm{ms}\ (\times 2\ \text{por periodo})\qquad V_{1,2} = \pm V_{OH}\dfrac{R_1}{R_F} = \pm 10{,}5\,\mathrm{V}\cdot\dfrac{10}{100} = \pm 1{,}05\,\mathrm{V}$$

In [ ]:
s = Esquema(17.0, 6.6)
cols = ['#EAF1F4', '#FCEFC7', '#EAF1F4', '#D7F0E1', '#F7D9D4']
items = [(0.3, 'Oscilador\nMiller-Schmitt\n±5 V, 1 Hz'), (3.7, 'Comparador\nde ventana\n±0,5 V'), (7.1, 'Restador\n$(v_A - v_B)/2$'), (10.5, 'Biestable\n(Schmitt sin\nentrada)'), (13.9, 'LED\n«GANASTE»')]
for (x, t), c in zip(items, cols):
    s.caja(x, 2.6, 2.8, 2.0, t, color=c, size=12)
for x in (3.1, 6.5, 9.9, 13.3):
    s.flecha((x, 3.6), (x + 0.6, 3.6))
s.texto((11.9, 2.15), 'botones JUGAR y REINICIAR', size=11, color='#4F606B')
s.texto((1.7, 5.3), 'lazo abierto + positiva', size=11, color='#4F606B')
s.texto((5.1, 5.3), 'lazo abierto', size=11, color='#4F606B')
s.texto((8.5, 5.3), 'realim. negativa', size=11, color='#4F606B')
s.texto((11.9, 5.3), 'realim. positiva', size=11, color='#4F606B')
s.texto((8.5, 0.8), 'Se gana si el botón se pulsa mientras la rampa está dentro de la ventana verde', size=12)
s.mostrar()

In [ ]:
s = Esquema(14.2, 8.0)
A = s.opamp(8.0, 3.4, mas_arriba=True, label='A6')
s.terminal((2.4, 3.85), r'$v_W$', size=13)
s.interruptor((2.4, 3.85), (4.4, 3.85), 'JUGAR')
s.nodo((4.4, 3.85)); s.resistencia((4.4, 3.85), (6.6, 3.85), r'$R_1$ 10 kΩ', lado=-1)
s.linea((6.6, 3.85), A['noinv']); s.nodo((7.0, 3.85)); s.nodo((7.5, 3.85))
s.resistencia((4.4, 3.85), (4.4, 1.0), '1 MΩ', lado=-1); s.tierra((4.4, 1.0))
s.terminal((2.4, 6.6), '−12 V', size=12); s.interruptor((2.4, 6.6), (4.6, 6.6), 'REINICIAR')
s.linea((4.6, 6.6), (7.0, 6.6), (7.0, 3.85))
s.linea((7.5, 3.85), (7.5, 5.3)); s.resistencia((7.5, 5.3), (10.8, 5.3), r'$R_F$ 100 kΩ'); s.linea((10.8, 5.3), (10.8, 3.4))
s.linea(A['inv'], (7.6, A['inv'][1])); s.tierra((7.6, A['inv'][1]))
s.linea(A['out'], (12.2, 3.4)); s.nodo((10.8, 3.4))
s.resistencia((12.2, 3.4), (12.2, 2.2), '820 Ω'); s.diodo((12.2, 2.2), (12.2, 1.0), None, led='verde'); s.tierra((12.2, 1.0))
s.texto((12.0, 0.2), 'GANASTE', ha='center', size=12, color=LED_COLORS['verde'], weight='bold')
s.mostrar()

In [ ]:
# Simulación: triángulo, ventana, señal v_W y estado del biestable con pulsaciones aleatorias
rng = np.random.default_rng(11)
f, W = 2.1 / (4 * 510e3 * 1e-6), 0.5
t = np.linspace(0, 4, 8000)
fase = (t * f) % 1
tri = np.where(fase < 0.5, -5 + 20 * fase, 15 - 20 * fase)
vW = np.where(np.abs(tri) < W, 10.5, 0.0)          # comparador de ventana + restador
pulsos = np.sort(rng.uniform(0.2, 3.8, 7))
estado, salida, latch = VOL, [], VOL
for ti in t:
    # mientras se pulsa (20 ms) la entrada del biestable es v_W; si no, 0 V por la resistencia de 1 MΩ
    vin = vW[np.searchsorted(t, ti)] if np.any(np.abs(pulsos - ti) < 0.01) else 0.0
    if latch < 0 and vin > 1.05: latch = VOH
    salida.append(latch)
fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
ax[0].plot(t, tri, color='#2F6DB5'); ax[0].axhspan(-W, W, color='#2E9B57', alpha=.2); ax[0].set_ylabel('triángulo (V)')
for p in pulsos: ax[0].axvline(p, color='#F2A33A', lw=1)
ax[1].plot(t, vW, color='#2E9B57'); ax[1].set_ylabel('$v_W$ (V)')
ax[2].plot(t, salida, color='#D2402F'); ax[2].set_ylabel('biestable (V)'); ax[2].set_xlabel('tiempo (s)')
for a in ax: a.grid(alpha=.25); a.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()
print('Las líneas naranjas son pulsaciones de JUGAR; el biestable solo cambia si coinciden con la ventana verde.')

### 4.3 Theremin óptico
La LDR (en serie con 4,7 kΩ) es la resistencia del integrador del Miller-Schmitt:

$$f = \dfrac{R_F}{4\,(R_{LDR}+4{,}7\,\mathrm{k\Omega})\,C\,R_1} = \dfrac{27\,\mathrm{k\Omega}}{4\,(R_{LDR}+4{,}7\,\mathrm{k\Omega})\cdot 47\,\mathrm{nF}\cdot 10\,\mathrm{k\Omega}}$$

In [ ]:
s = Esquema(16.0, 7.6)
A1 = s.opamp(6, 4, label='A1')
s.linea(A1['inv'], (5.4, A1['inv'][1])); s.linea((5.4, A1['inv'][1]), (5.4, 6.6))
s.condensador((5.4, 6.6), (8.4, 6.6), r'$C$ 47 nF'); s.linea((8.4, 6.6), (8.4, 4.0)); s.linea(A1['out'], (8.4, 4.0))
s.linea(A1['noinv'], (5.6, A1['noinv'][1])); s.tierra((5.6, A1['noinv'][1]))
s.ldr((2.0, 4.45), (4.4, 4.45), 'LDR', lado=1)
s.resistencia((4.4, 4.45), (5.4, 4.45), None, cuerpo=0.5); s.texto((4.9, 3.9), '4,7 kΩ', size=10)
s.nodo((5.4, 4.45))
A2 = s.opamp(11.2, 4, mas_arriba=True, label='A2')
s.nodo((8.4, 4.45)); s.resistencia((8.4, 4.45), (10.4, 4.45), r'$R_1$ 10 kΩ', lado=-1)
s.linea((10.4, 4.45), A2['noinv']); s.nodo((10.6, 4.45))
s.linea((10.6, 4.45), (10.6, 5.7)); s.resistencia((10.6, 5.7), (13.6, 5.7), r'$R_F$ 27 kΩ'); s.linea((13.6, 5.7), (13.6, 4.0))
s.linea(A2['inv'], (10.8, A2['inv'][1])); s.tierra((10.8, A2['inv'][1]))
s.linea(A2['out'], (14.2, 4.0)); s.nodo((13.6, 4.0))
s.resistencia((14.2, 4.0), (14.2, 2.4), '1 kΩ', lado=-1); s.linea((14.2, 2.4), (14.6, 2.4)); s.parlante((14.6, 2.4), 'piezo')
s.linea((13.6, 4.0), (13.6, 1.0), (2.0, 1.0), (2.0, 4.45))
s.mostrar()

# theremin: f vs luz
Rl = np.logspace(3.3, 6, 200)
f = 27e3 / (4 * (Rl + 4.7e3) * 47e-9 * 10e3)
fig, ax = plt.subplots(figsize=(6.2, 4.0))
ax.loglog(Rl, f, color='#2F6DB5', lw=2.4)
ax.set_xlabel(r'$R_{LDR}$ (Ω)  ·  mucha luz ← → oscuridad'); ax.set_ylabel('frecuencia (Hz)'); ax.grid(alpha=.3, which='both')
for R, lab in ((5e3, '5 kΩ'), (50e3, '50 kΩ'), (500e3, '500 kΩ')):
    fr = 27e3 / (4 * (R + 4.7e3) * 47e-9 * 10e3); ax.plot(R, fr, 'o', color=LED_COLORS['rojo']); ax.text(R * 1.15, fr * 1.1, f'{fr:,.0f} Hz'.replace(',', '.'), fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout(); plt.show()

In [ ]:
# Escuche el theremin: la mano pasa de la luz (5 kΩ) a la sombra (500 kΩ) y regresa
from IPython.display import Audio, display
fs = 22050
t = np.arange(0, 5, 1 / fs)
R_ldr = 5e3 * (100 ** (0.5 - 0.5 * np.cos(2 * np.pi * t / 5)))      # 5 kΩ → 500 kΩ → 5 kΩ
f_inst = 27e3 / (4 * (R_ldr + 4.7e3) * 47e-9 * 10e3)
fase = 2 * np.pi * np.cumsum(f_inst) / fs
onda = 0.25 * np.sign(np.sin(fase))                                   # salida cuadrada del Schmitt
print(f'frecuencia: de {coma(f_inst.max(), 0)} Hz a {coma(f_inst.min(), 0)} Hz')
display(Audio(onda, rate=fs))

## 5. Página web con los ejemplos funcionales
La celda siguiente guarda la página como `operacionales_semana3.html` y la despliega aquí mismo, en un marco de 900 px con su propia barra de desplazamiento (use la rueda del mouse con el puntero sobre la página). Incluye: semáforo animado, comparador frente a Schmitt, D/A con interruptores, restador y multientradas, termostato, juego de reflejos (barra espaciadora) y theremin con sonido.

In [ ]:
# Página web autónoma (HTML + CSS + JavaScript)
PAGINA = r'''<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Operacionales en acción</title>
<style>
:root{
  --bg:#F4F7F8; --card:#FFFFFF; --ink:#1F2A33; --muted:#5B6B75; --line:#D5DEE2;
  --petrol:#12343F; --teal:#1E7F86; --amber:#F2A33A; --red:#D2402F; --green:#2E9B57; --yellow:#E8A317; --blue:#2F6DB5;
  --grid:#E3E9EC; --off:#C9D2D6;
}
@media (prefers-color-scheme: dark){
  :root:not([data-theme="light"]){
    --bg:#0F1E24; --card:#16303A; --ink:#E6EEF1; --muted:#9FB3BB; --line:#2A4651;
    --petrol:#0B2129; --grid:#24414B; --off:#3A5560;
  }
}
:root[data-theme="dark"]{ --bg:#0F1E24; --card:#16303A; --ink:#E6EEF1; --muted:#9FB3BB; --line:#2A4651; --petrol:#0B2129; --grid:#24414B; --off:#3A5560; }
*{box-sizing:border-box}
html{overflow-y:auto;height:auto}
body{margin:0;overflow-y:auto;min-height:100%;background:var(--bg);color:var(--ink);font-family:Calibri,"Segoe UI",system-ui,-apple-system,sans-serif;font-size:16px;line-height:1.45}
header{background:#12343F;color:#fff;padding:22px 20px 0}
header .k{font-size:12px;letter-spacing:2px;text-transform:uppercase;color:#F2A33A;font-weight:700}
header h1{font-family:Cambria,Georgia,serif;font-size:28px;margin:4px 0 14px}
nav{display:flex;gap:4px;overflow-x:auto;scrollbar-width:thin}
nav button{flex:0 0 auto;border:0;background:transparent;color:#C9D8DD;font:inherit;font-size:15px;padding:10px 14px;border-radius:10px 10px 0 0;cursor:pointer}
nav button[aria-selected="true"]{background:var(--bg);color:var(--ink);font-weight:700}
main{max-width:1180px;margin:0 auto;padding:18px 16px 40px}
section{display:none}
section.on{display:block}
h2{font-family:Cambria,Georgia,serif;font-size:24px;margin:4px 0 4px}
.lead{color:var(--muted);margin:0 0 14px}
.grid{display:grid;grid-template-columns:minmax(0,1.6fr) minmax(0,1fr);gap:16px}
@media (max-width:820px){.grid{grid-template-columns:1fr}}
.card{background:var(--card);border:1px solid var(--line);border-radius:14px;padding:14px 16px}
.card h3{margin:0 0 8px;font-size:13px;letter-spacing:1.5px;text-transform:uppercase;color:var(--teal)}
canvas{width:100%;display:block;border-radius:10px;background:var(--card)}
.ctl{display:grid;grid-template-columns:1fr auto;gap:2px 10px;align-items:center;margin:8px 0}
.ctl label{font-size:14px}
.ctl output{font-variant-numeric:tabular-nums;font-weight:700;font-size:14px}
.ctl input[type=range]{grid-column:1/3;width:100%;accent-color:var(--teal)}
.kv{display:grid;grid-template-columns:1fr auto;gap:4px 10px;font-size:15px}
.kv b{font-variant-numeric:tabular-nums}
.eq{font-family:Cambria,Georgia,serif;font-size:17px;background:var(--bg);border-radius:8px;padding:8px 10px;margin:8px 0}
.leds{display:flex;gap:18px;justify-content:center;padding:10px 0}
.led{width:58px;height:58px;border-radius:50%;background:var(--off);border:3px solid var(--line);transition:background .08s, box-shadow .08s}
.led.r.on{background:var(--red);box-shadow:0 0 22px var(--red)}
.led.y.on{background:var(--yellow);box-shadow:0 0 22px var(--yellow)}
.led.g.on{background:var(--green);box-shadow:0 0 22px var(--green)}
.btn{font:inherit;border:0;border-radius:10px;padding:9px 16px;background:var(--teal);color:#fff;cursor:pointer;font-weight:700}
.btn.alt{background:var(--amber);color:#12343F}
.btn.ghost{background:transparent;border:1px solid var(--line);color:var(--ink)}
.row{display:flex;gap:10px;flex-wrap:wrap;align-items:center}
.bits{display:flex;gap:10px}
.bit{width:64px;height:64px;border-radius:12px;border:2px solid var(--line);background:var(--bg);font:inherit;font-size:22px;font-weight:700;color:var(--ink);cursor:pointer}
.bit.on{background:var(--teal);color:#fff;border-color:var(--teal)}
.big{font-size:40px;font-weight:700;font-variant-numeric:tabular-nums;font-family:Cambria,Georgia,serif}
.note{font-size:14px;color:var(--muted)}
.tag{display:inline-block;font-size:12px;border-radius:999px;padding:2px 10px;background:var(--bg);border:1px solid var(--line);margin-right:6px}
.win{color:var(--green);font-weight:700}.lose{color:var(--red);font-weight:700}
.esq{margin:0 0 16px}
.esq .figs{display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:14px;align-items:start}
.esq figure{margin:0}
.esq img{width:100%;height:auto;max-height:340px;object-fit:contain;background:#fff;border-radius:10px;padding:8px;border:1px solid var(--line)}
.esq figcaption{font-size:13px;color:var(--muted);margin-top:6px}
.esq .vals{font-size:14px;margin-top:10px;font-variant-numeric:tabular-nums}
.esq summary{cursor:pointer;font-size:13px;letter-spacing:1.5px;text-transform:uppercase;color:var(--teal);font-weight:700;list-style:none}
.esq summary::-webkit-details-marker{display:none}
.esq summary::before{content:'▾ ';}
.esq details:not([open]) summary::before{content:'▸ ';}
footer{max-width:1180px;margin:0 auto;padding:0 16px 30px;color:var(--muted);font-size:13px}
</style>
</head>
<body>
<header>
  <div class="k">Semana 3 · Circuitos eléctricos y microcontroladores</div>
  <h1>El amplificador operacional en acción</h1>
  <nav role="tablist" id="tabs">
    <button role="tab" data-t="semaforo" aria-selected="true">Semáforo</button>
    <button role="tab" data-t="comparador">Comparador vs Schmitt</button>
    <button role="tab" data-t="dac">Sumador D/A</button>
    <button role="tab" data-t="restador">Restador y multientradas</button>
    <button role="tab" data-t="termo">Termostato</button>
    <button role="tab" data-t="juego">Juego de reflejos</button>
    <button role="tab" data-t="theremin">Theremin</button>
  </nav>
</header>
<main>

<!-- SEMÁFORO -->
<section id="semaforo" class="on">
  <h2>Semáforo solo con operacionales</h2>
  <p class="lead">Integrador + Schmitt generan la rampa; tres comparadores en lazo abierto encienden cada LED. Modifique los componentes y observe los tiempos.</p>
  <div class="card esq"><details open><summary>Esquema del circuito</summary><div class="figs" style="margin-top:10px"><figure><img alt="Generador de rampa" src="..."><figcaption>Generador de rampa: A1 integra, A2 (Schmitt) decide el fin de la rampa; D1/R<sub>u</sub> suben lento y D2/R<sub>d</sub> bajan rápido.</figcaption></figure><figure><img alt="Comparadores y LEDs" src="..."><figcaption>Escalera R<sub>a</sub>–R<sub>b</sub>–R<sub>c</sub> y comparadores C1–C3 en lazo abierto; el LED amarillo va entre las salidas de C2 y C1.</figcaption></figure></div><div class="vals" id="sfVals"></div></details></div>
  <div class="grid">
    <div class="card">
      <h3>Rampa v<sub>O1</sub> y umbrales</h3>
      <canvas id="sfCan" height="300"></canvas>
      <div class="leds"><div class="led g" id="lg"></div><div class="led y" id="ly"></div><div class="led r" id="lr"></div></div>
    </div>
    <div class="card">
      <h3>Componentes</h3>
      <div class="ctl"><label for="sfRu">R<sub>u</sub> (subida)</label><output id="sfRuO"></output><input type="range" id="sfRu" min="0.5" max="3" step="0.1" value="1.5"></div>
      <div class="ctl"><label for="sfC">C</label><output id="sfCO"></output><input type="range" id="sfC" min="4.7" max="22" step="0.1" value="10"></div>
      <div class="ctl"><label for="sfVA">V<sub>A</sub> (verde → amarillo)</label><output id="sfVAO"></output><input type="range" id="sfVA" min="2" max="8" step="0.05" value="5.1"></div>
      <div class="ctl"><label for="sfVB">V<sub>B</sub> (amarillo → rojo)</label><output id="sfVBO"></output><input type="range" id="sfVB" min="2.2" max="8.8" step="0.05" value="6.4"></div>
      <div class="ctl"><label for="sfSp">Velocidad de simulación</label><output id="sfSpO"></output><input type="range" id="sfSp" min="1" max="8" step="1" value="2"></div>
      <div class="eq">dv/dt = (|V<sub>OL</sub>| − V<sub>D</sub>) / (R<sub>u</sub>·C)</div>
      <div class="kv">
        <span>Pendiente</span><b id="sfP"></b>
        <span>Verde</span><b id="sfTg"></b>
        <span>Amarillo</span><b id="sfTy"></b>
        <span>Rojo</span><b id="sfTr"></b>
        <span>Ciclo completo</span><b id="sfT"></b>
      </div>
      <p class="note">V<sub>1</sub> = 8,89 V y V<sub>2</sub> = 1,12 V (Schmitt con R<sub>1</sub> = 10 kΩ, R<sub>F</sub> = 27 kΩ, V<sub>r</sub> = 3,65 V). Retorno por R<sub>d</sub> = 12 kΩ.</p>
    </div>
  </div>
</section>

<!-- COMPARADOR -->
<section id="comparador">
  <h2>Comparador en lazo abierto frente a Schmitt</h2>
  <p class="lead">Una señal lenta con ruido cruza el umbral. Sin histéresis la salida rebota; con realimentación positiva conmuta una sola vez.</p>
  <div class="card esq"><details open><summary>Esquema del circuito</summary><div class="figs" style="margin-top:10px"><figure><img alt="Comparador" src="..."><figcaption>Comparador en lazo abierto: referencia por divisor de 10 kΩ y 4,7 kΩ.</figcaption></figure><figure><img alt="Schmitt no inversor" src="..."><figcaption>Schmitt no inversor: la misma referencia en la entrada − y realimentación positiva por R<sub>F</sub>.</figcaption></figure></div><div class="vals" id="cmVals"></div></details></div>
  <div class="grid">
    <div class="card"><h3>Entrada y salidas</h3><canvas id="cmCan" height="380"></canvas></div>
    <div class="card">
      <h3>Parámetros</h3>
      <div class="ctl"><label for="cmN">Ruido (pico)</label><output id="cmNO"></output><input type="range" id="cmN" min="0" max="1" step="0.02" value="0.3"></div>
      <div class="ctl"><label for="cmRF">R<sub>F</sub> del Schmitt</label><output id="cmRFO"></output><input type="range" id="cmRF" min="20" max="500" step="5" value="100"></div>
      <div class="eq">V<sub>H</sub> = (V<sub>OH</sub> − V<sub>OL</sub>)·R<sub>1</sub>/R<sub>F</sub><br>V<sub>1,2</sub> = V<sub>ref</sub>(1 + R<sub>1</sub>/R<sub>F</sub>) − V<sub>OL,OH</sub>·R<sub>1</sub>/R<sub>F</sub></div>
      <div class="kv">
        <span>V<sub>ref</sub></span><b>3,84 V</b>
        <span>Histéresis V<sub>H</sub></span><b id="cmVH"></b>
        <span>Conmutaciones (comparador)</span><b id="cmK1"></b>
        <span>Conmutaciones (Schmitt)</span><b id="cmK2"></b>
      </div>
      <p class="row" style="margin-top:12px"><button class="btn ghost" id="cmNew">Nueva señal</button></p>
      <p class="note">Regla práctica: V<sub>H</sub> debe superar el ruido pico a pico.</p>
    </div>
  </div>
</section>

<!-- DAC -->
<section id="dac">
  <h2>Sumador inversor: convertidor D/A de 3 bits</h2>
  <p class="lead">Cada bit (0 o 5 V) entra por una resistencia de peso binario: 10 kΩ, 20 kΩ y 40 kΩ. A2 invierte para obtener una salida positiva.</p>
  <div class="card esq"><details open><summary>Esquema del circuito</summary><div class="figs" style="margin-top:10px"><figure><img alt="Convertidor D/A" src="..."><figcaption>Sumador inversor A1 con resistencias ponderadas e inversor A2 de ganancia −1.</figcaption></figure></div><div class="vals" id="dacVals"></div></details></div>
  <div class="grid">
    <div class="card">
      <h3>Bits de entrada</h3>
      <div class="row" style="justify-content:space-between">
        <div class="bits"><button class="bit" data-b="2">0</button><button class="bit" data-b="1">0</button><button class="bit" data-b="0">0</button></div>
        <div><div class="note">v<sub>O</sub> (salida de A2)</div><div class="big" id="dacV">0,000 V</div></div>
      </div>
      <canvas id="dacCan" height="260" style="margin-top:12px"></canvas>
    </div>
    <div class="card">
      <h3>Diseño</h3>
      <div class="ctl"><label for="dacRF">R<sub>F</sub></label><output id="dacRFO"></output><input type="range" id="dacRF" min="4" max="12" step="0.1" value="8"></div>
      <div class="eq">v<sub>O1</sub> = −R<sub>F</sub>(v<sub>b2</sub>/10k + v<sub>b1</sub>/20k + v<sub>b0</sub>/40k)<br>v<sub>O</sub> = −v<sub>O1</sub></div>
      <div class="kv">
        <span>Código N</span><b id="dacN"></b>
        <span>v<sub>O1</sub></span><b id="dacV1"></b>
        <span>Paso (LSB)</span><b id="dacL"></b>
        <span>Corriente en R<sub>F</sub></span><b id="dacI"></b>
      </div>
      <p class="note">Si R<sub>F</sub> es muy grande, v<sub>O1</sub> llega a −10,5 V y la salida se satura: observe el código 111 con R<sub>F</sub> = 12 kΩ.</p>
    </div>
  </div>
</section>

<!-- RESTADOR -->
<section id="restador">
  <h2>Restador diferencial y multientradas</h2>
  <p class="lead">Dos aplicaciones de la realimentación negativa: medir corriente con un shunt y combinar tres señales con signos distintos.</p>
  <div class="card esq"><details open><summary>Esquema del circuito</summary><div class="figs" style="margin-top:10px"><figure><img alt="Restador diferencial" src="..."><figcaption>Medidor de corriente: R<sub>S</sub> = 0,1 Ω y restador de ganancia R<sub>2</sub>/R<sub>1</sub> = 16,5.</figcaption></figure><figure><img alt="Multientradas" src="..."><figcaption>Multientradas universal: v<sub>O</sub> = 2v<sub>1</sub> − 3v<sub>2</sub> + v<sub>3</sub> con R<sub>F</sub> = 60 kΩ.</figcaption></figure></div></details></div>
  <div class="grid" style="grid-template-columns:1fr 1fr">
    <div class="card">
      <h3>Medidor de corriente (G = 16,5)</h3>
      <div class="ctl"><label for="rsI">Corriente del motor I</label><output id="rsIO"></output><input type="range" id="rsI" min="0" max="2.5" step="0.01" value="1.2"></div>
      <div class="ctl"><label for="rsT">Error de las resistencias ε</label><output id="rsTO"></output><input type="range" id="rsT" min="0" max="5" step="0.1" value="1"></div>
      <div class="ctl"><label for="rsCM">Modo común v<sub>1</sub> (cable de tierra largo)</label><output id="rsCMO"></output><input type="range" id="rsCM" min="0" max="1" step="0.01" value="0.2"></div>
      <div class="eq">v<sub>O</sub> = (R<sub>2</sub>/R<sub>1</sub>)(v<sub>2</sub> − v<sub>1</sub>) = 16,5·R<sub>S</sub>·I</div>
      <div class="kv">
        <span>Ideal</span><b id="rsVi"></b>
        <span>Con desapareamiento (peor caso)</span><b id="rsVr"></b>
        <span>Error</span><b id="rsE"></b>
        <span>CMRR estimado</span><b id="rsC"></b>
      </div>
      <canvas id="rsCan" height="70" style="margin-top:10px"></canvas>
    </div>
    <div class="card">
      <h3>Multientradas: v<sub>O</sub> = 2v<sub>1</sub> − 3v<sub>2</sub> + v<sub>3</sub></h3>
      <div class="ctl"><label for="u1">v<sub>1</sub></label><output id="u1O"></output><input type="range" id="u1" min="-5" max="5" step="0.05" value="1"></div>
      <div class="ctl"><label for="u2">v<sub>2</sub></label><output id="u2O"></output><input type="range" id="u2" min="-5" max="5" step="0.05" value="0.5"></div>
      <div class="ctl"><label for="u3">v<sub>3</sub></label><output id="u3O"></output><input type="range" id="u3" min="-5" max="5" step="0.05" value="2"></div>
      <div class="eq">R<sub>F</sub> = 60 kΩ · v<sub>1</sub>: 30 kΩ · v<sub>3</sub>: 60 kΩ · R<sub>aux+</sub> = 60 kΩ · v<sub>2</sub>: 20 kΩ</div>
      <div class="kv">
        <span>v<sub>+</sub> = (2v<sub>1</sub> + v<sub>3</sub>)/4</span><b id="uVp"></b>
        <span>v<sub>O</sub> calculada</span><b id="uVc"></b>
        <span>v<sub>O</sub> real (con saturación)</span><b id="uVo"></b>
      </div>
      <p class="note" id="uSat"></p>
    </div>
  </div>
</section>

<!-- TERMOSTATO -->
<section id="termo">
  <h2>Termostato: ON/OFF con Schmitt frente a control proporcional</h2>
  <p class="lead">Horno con constante de tiempo de 120 s y sensor con retardo de 15 s. El Schmitt decide con dos umbrales; el control P regula con realimentación negativa.</p>
  <div class="card esq"><details open><summary>Esquema del circuito</summary><div class="figs" style="margin-top:10px"><figure><img alt="Termostato" src="..."><figcaption>LM35 × 10 y Schmitt inversor: R<sub>A</sub>, R<sub>B</sub> fijan la referencia y R<sub>F</sub> la histéresis.</figcaption></figure></div><div class="vals" id="tmVals"></div></details></div>
  <div class="grid">
    <div class="card"><h3>Temperatura (simulación acelerada)</h3><canvas id="tmCan" height="320"></canvas>
      <div class="row" style="margin-top:8px"><span class="tag" style="border-color:var(--red)">ON/OFF</span><span class="tag" style="border-color:var(--blue)">Proporcional</span><span class="note" id="tmState"></span></div>
    </div>
    <div class="card">
      <h3>Ajustes</h3>
      <div class="ctl"><label for="tmSP">Consigna</label><output id="tmSPO"></output><input type="range" id="tmSP" min="25" max="45" step="0.5" value="30"></div>
      <div class="ctl"><label for="tmH">Histéresis total (Schmitt)</label><output id="tmHO"></output><input type="range" id="tmH" min="0" max="6" step="0.1" value="2"></div>
      <div class="ctl"><label for="tmK">K<sub>p</sub> (control P)</label><output id="tmKO"></output><input type="range" id="tmK" min="0.1" max="3" step="0.05" value="0.5"></div>
      <div class="ctl"><label for="tmN">Ruido del sensor</label><output id="tmNO"></output><input type="range" id="tmN" min="0" max="0.5" step="0.01" value="0.1"></div>
      <div class="kv">
        <span>Conmutaciones del relé</span><b id="tmSw"></b>
        <span>Temperatura ON/OFF</span><b id="tmT1"></b>
        <span>Temperatura P</span><b id="tmT2"></b>
        <span>R<sub>F</sub> equivalente (R<sub>1</sub> = 1 kΩ)</span><b id="tmRF"></b>
      </div>
      <p class="row" style="margin-top:12px"><button class="btn ghost" id="tmReset">Reiniciar</button></p>
      <p class="note">Con histéresis 0 y ruido, el relé conmuta cientos de veces: así se ve por qué el termostato necesita realimentación positiva.</p>
    </div>
  </div>
</section>

<!-- JUEGO -->
<section id="juego">
  <h2>Juego de reflejos: detén la luz en la zona verde</h2>
  <p class="lead">Un Miller-Schmitt genera un triángulo; un comparador de ventana detecta la zona; al pulsar, un Schmitt biestable guarda el resultado. Pulse el botón o la barra espaciadora.</p>
  <div class="card esq"><details open><summary>Esquema del circuito</summary><div class="figs" style="margin-top:10px"><figure><img alt="Diagrama de bloques del juego" src="..."><figcaption>Cadena completa: oscilador, ventana, restador y biestable.</figcaption></figure><figure><img alt="Biestable" src="..."><figcaption>Memoria: Schmitt no inversor con botones JUGAR y REINICIAR.</figcaption></figure></div><div class="vals" id="jgVals"></div></details></div>
  <div class="grid">
    <div class="card">
      <h3>Triángulo y ventana</h3>
      <canvas id="jgCan" height="300" tabindex="0" aria-label="Zona de juego"></canvas>
      <div class="row" style="margin-top:10px;justify-content:space-between">
        <button class="btn alt" id="jgBtn">JUGAR</button>
        <button class="btn ghost" id="jgRst">REINICIAR</button>
        <div class="leds" style="padding:0"><div class="led g" id="jgLed" style="width:44px;height:44px"></div></div>
        <div id="jgMsg" class="note" style="min-width:150px"></div>
      </div>
    </div>
    <div class="card">
      <h3>Dificultad</h3>
      <div class="ctl"><label for="jgR">R del integrador</label><output id="jgRO"></output><input type="range" id="jgR" min="100" max="1000" step="10" value="510"></div>
      <div class="ctl"><label for="jgW">Ventana ±</label><output id="jgWO"></output><input type="range" id="jgW" min="0.1" max="2" step="0.05" value="0.5"></div>
      <div class="eq">f = R<sub>F</sub>/(4·R·C·R<sub>1</sub>), C = 1 µF, R<sub>F</sub>/R<sub>1</sub> = 2,1</div>
      <div class="kv">
        <span>Frecuencia</span><b id="jgF"></b>
        <span>Tiempo dentro de la ventana</span><b id="jgTw"></b>
        <span>Aciertos / intentos</span><b id="jgS"></b>
        <span>Racha</span><b id="jgSt"></b>
      </div>
      <p class="note">El biestable (umbrales ±1,05 V) solo cambia si la señal de la ventana vale 10,5 V en el instante de pulsar.</p>
    </div>
  </div>
</section>

<!-- THEREMIN -->
<section id="theremin">
  <h2>Theremin óptico</h2>
  <p class="lead">Miller-Schmitt con una LDR como resistencia del integrador. Mueva el puntero (o el dedo) sobre el panel: arriba hay más luz y el tono sube.</p>
  <div class="card esq"><details open><summary>Esquema del circuito</summary><div class="figs" style="margin-top:10px"><figure><img alt="Theremin óptico" src="..."><figcaption>Miller-Schmitt: la LDR + 4,7 kΩ es la resistencia del integrador; la salida cuadrada va al piezo.</figcaption></figure></div><div class="vals" id="thVals"></div></details></div>
  <div class="grid">
    <div class="card">
      <h3>Panel de luz</h3>
      <canvas id="thCan" height="300" style="touch-action:none;cursor:crosshair"></canvas>
      <div class="row" style="margin-top:10px"><button class="btn" id="thOn">Activar sonido</button><span class="note">El navegador exige un clic antes de reproducir audio.</span></div>
    </div>
    <div class="card">
      <h3>Lectura</h3>
      <div class="ctl"><label for="thL">Iluminación</label><output id="thLO"></output><input type="range" id="thL" min="0" max="100" step="1" value="50"></div>
      <div class="eq">f = R<sub>F</sub> / [4(R<sub>LDR</sub> + 4,7 kΩ)·C·R<sub>1</sub>]<br>C = 47 nF, R<sub>F</sub> = 27 kΩ, R<sub>1</sub> = 10 kΩ</div>
      <div class="kv">
        <span>R<sub>LDR</sub></span><b id="thR"></b>
        <span>Frecuencia</span><b id="thF"></b>
        <span>Amplitud del triángulo</span><b>±3,89 V</b>
      </div>
      <canvas id="thW" height="120" style="margin-top:10px"></canvas>
    </div>
  </div>
</section>
</main>
<footer>Modelo usado: TL084 a ±12 V, V<sub>OH</sub> = +10,5 V, V<sub>OL</sub> = −10,5 V, V<sub>D</sub> = 0,6 V. Página autónoma: funciona sin conexión y dentro de Google Colab.</footer>

<script>
(function(){
'use strict';
const VOH = 10.5, VOL = -10.5, VD = 0.6;
const fmt = (x, d=2) => x.toFixed(d).replace('.', ',');
const $ = (id) => document.getElementById(id);
const css = (v) => getComputedStyle(document.documentElement).getPropertyValue(v).trim();

// ---------- pestañas ----------
let active = 'semaforo';
document.querySelectorAll('#tabs button').forEach(b => b.addEventListener('click', () => {
  document.querySelectorAll('#tabs button').forEach(x => x.setAttribute('aria-selected', x === b));
  document.querySelectorAll('main section').forEach(s => s.classList.toggle('on', s.id === b.dataset.t));
  active = b.dataset.t;
  resizeAll();
}));

// ---------- utilidades de canvas ----------
function fit(c){
  const r = c.getBoundingClientRect(); const dpr = window.devicePixelRatio || 1;
  const h = +c.getAttribute('height');
  if (!c._h) c._h = h;
  c.style.height = c._h + 'px';
  c.width = Math.max(10, Math.round(r.width * dpr)); c.height = Math.round(c._h * dpr);
  const g = c.getContext('2d'); g.setTransform(dpr, 0, 0, dpr, 0, 0);
  return { g, w: r.width, h: c._h };
}
function axes(g, w, h, m, x0, x1, y0, y1, xl, yl, ticksY){
  g.fillStyle = css('--card'); g.fillRect(0, 0, w, h);
  g.strokeStyle = css('--grid'); g.lineWidth = 1; g.font = '12px Calibri, sans-serif'; g.fillStyle = css('--muted');
  const X = (x) => m.l + (x - x0) / (x1 - x0) * (w - m.l - m.r);
  const Y = (y) => h - m.b - (y - y0) / (y1 - y0) * (h - m.t - m.b);
  (ticksY || []).forEach(t => { g.beginPath(); g.moveTo(m.l, Y(t)); g.lineTo(w - m.r, Y(t)); g.stroke(); g.fillText(fmt(t, Math.abs(t) < 10 && t % 1 ? 1 : 0), 4, Y(t) + 4); });
  g.strokeStyle = css('--line'); g.beginPath(); g.moveTo(m.l, m.t); g.lineTo(m.l, h - m.b); g.lineTo(w - m.r, h - m.b); g.stroke();
  if (xl) { g.fillText(xl, w - m.r - g.measureText(xl).width, h - 6); }
  if (yl) { g.fillText(yl, m.l + 4, m.t - 4 > 10 ? m.t - 4 : 12); }
  return { X, Y };
}
function line(g, pts, X, Y, color, lw=2){
  g.strokeStyle = color; g.lineWidth = lw; g.beginPath();
  pts.forEach((p, i) => i ? g.lineTo(X(p[0]), Y(p[1])) : g.moveTo(X(p[0]), Y(p[1]))); g.stroke();
}
function hline(g, y, X0, X1, Y, color, label){
  g.strokeStyle = color; g.setLineDash([5, 4]); g.lineWidth = 1.3; g.beginPath(); g.moveTo(X0, Y(y)); g.lineTo(X1, Y(y)); g.stroke(); g.setLineDash([]);
  if (label) { g.fillStyle = color; g.font = '12px Calibri, sans-serif'; g.fillText(label, X1 + 4, Y(y) + 4); }
}
function bindRange(id, f, cb){
  const el = $(id), out = $(id + 'O');
  const upd = () => { if (out) out.textContent = f(+el.value); cb && cb(); };
  el.addEventListener('input', upd); if (out) out.textContent = f(+el.value); return el;
}

// ================= SEMÁFORO =================
const SF = { R1: 10e3, RF: 27e3, Vr: 12 * 10.5 / 34.5, Rd: 12e3 };
SF.V1 = SF.Vr * (1 + SF.R1 / SF.RF) - VOL * SF.R1 / SF.RF;
SF.V2 = SF.Vr * (1 + SF.R1 / SF.RF) - VOH * SF.R1 / SF.RF;
let sf = { v: SF.V2, out: VOL, t: 0, hist: [] };
const sfRu = bindRange('sfRu', v => fmt(v, 1) + ' MΩ', sfCalc);
const sfC = bindRange('sfC', v => fmt(v, 1) + ' µF', sfCalc);
const sfVA = bindRange('sfVA', v => fmt(v, 2) + ' V', () => { if (+sfVA.value > +$('sfVB').value - 0.2) $('sfVB').value = +sfVA.value + 0.2; sfCalc(); });
const sfVB = bindRange('sfVB', v => fmt(v, 2) + ' V', () => { if (+sfVB.value < +sfVA.value + 0.2) sfVA.value = +sfVB.value - 0.2; sfCalc(); });
bindRange('sfSp', v => '×' + v);
function sfCalc(){
  const Ru = +sfRu.value * 1e6, C = +sfC.value * 1e-6, VA = +sfVA.value, VB = +$('sfVB').value;
  $('sfVAO').textContent = fmt(VA) + ' V'; $('sfVBO').textContent = fmt(VB) + ' V';
  const p = (-VOL - VD) / (Ru * C);
  const clamp = (x) => Math.max(0, x);
  const tg = clamp(Math.min(VA, SF.V1) - SF.V2) / p, ty = clamp(Math.min(VB, SF.V1) - Math.max(VA, SF.V2)) / p, tr = clamp(SF.V1 - Math.max(VB, SF.V2)) / p;
  const tb = (SF.V1 - SF.V2) * SF.Rd * C / (VOH - VD);
  $('sfP').textContent = fmt(p, 3) + ' V/s'; $('sfTg').textContent = fmt(tg, 2) + ' s'; $('sfTy').textContent = fmt(ty, 2) + ' s';
  $('sfTr').textContent = fmt(tr, 2) + ' s'; $('sfT').textContent = fmt(tg + ty + tr + tb, 2) + ' s';
  $('sfVals').innerHTML = 'Valores actuales: R<sub>u</sub> = ' + fmt(Ru / 1e6, 1) + ' MΩ · C = ' + fmt(C * 1e6, 1) + ' µF · escalera con 1 mA: R<sub>a</sub> = ' + fmt(12 - VB, 2) + ' kΩ, R<sub>b</sub> = ' + fmt(VB - VA, 2) + ' kΩ, R<sub>c</sub> = ' + fmt(VA, 2) + ' kΩ';
}
function sfStep(dt){
  const Ru = +sfRu.value * 1e6, C = +sfC.value * 1e-6, n = 40, h = dt / n;
  for (let i = 0; i < n; i++) {
    if (sf.out < 0) sf.v += (-VOL - VD) / (Ru * C) * h; else sf.v -= (VOH - VD) / (SF.Rd * C) * h;
    const vp = sf.v * SF.RF / (SF.R1 + SF.RF) + sf.out * SF.R1 / (SF.R1 + SF.RF);
    if (sf.out < 0 && vp > SF.Vr) sf.out = VOH; else if (sf.out > 0 && vp < SF.Vr) sf.out = VOL;
  }
  sf.t += dt; sf.hist.push([sf.t, sf.v]);
  while (sf.hist.length && sf.hist[0][0] < sf.t - 30) sf.hist.shift();
}
function sfDraw(){
  const { g, w, h } = fit($('sfCan')); const VA = +sfVA.value, VB = +$('sfVB').value;
  const m = { l: 34, r: 70, t: 14, b: 22 };
  const t1 = Math.max(30, sf.t), t0 = t1 - 30;
  const { X, Y } = axes(g, w, h, m, t0, t1, 0, 10, 'tiempo (s)', 'V', [0, 2, 4, 6, 8, 10]);
  hline(g, SF.V1, m.l, w - m.r, Y, css('--muted'), 'V₁ ' + fmt(SF.V1));
  hline(g, VB, m.l, w - m.r, Y, css('--red'), 'V_B ' + fmt(VB));
  hline(g, VA, m.l, w - m.r, Y, css('--green'), 'V_A ' + fmt(VA));
  hline(g, SF.V2, m.l, w - m.r, Y, css('--muted'), 'V₂ ' + fmt(SF.V2));
  line(g, sf.hist, X, Y, css('--blue'), 2.2);
  const v = sf.v, c1 = v > VB, c2 = v > VA, c3 = v < VA;
  $('lg').classList.toggle('on', c3); $('ly').classList.toggle('on', c2 && !c1); $('lr').classList.toggle('on', c1);
}

// ================= COMPARADOR =================
let cmSeed = 1, cmNoise = [];
function rnd(){ cmSeed = (cmSeed * 16807) % 2147483647; return cmSeed / 2147483647; }
function cmGen(){ cmNoise = []; let a = 0; for (let i = 0; i <= 600; i++) { a = 0.6 * a + 0.4 * (rnd() * 2 - 1); cmNoise.push(a); } }
cmGen();
bindRange('cmN', v => fmt(v, 2) + ' V', cmDraw);
bindRange('cmRF', v => v + ' kΩ', cmDraw);
$('cmNew').addEventListener('click', () => { cmSeed = Math.floor(Math.random() * 1e6) + 1; cmGen(); cmDraw(); });
function cmDraw(){
  if (active !== 'comparador') return;
  const { g, w, h } = fit($('cmCan')); const N = +$('cmN').value, RF = +$('cmRF').value * 1e3, R1 = 10e3, Vref = 3.84;
  const kk = R1 / RF, V1 = Vref * (1 + kk) - VOL * kk, V2 = Vref * (1 + kk) - VOH * kk, VH = V1 - V2;
  const vin = [], o1 = [], o2 = []; let s2 = VOL, k1 = 0, k2 = 0, p1 = VOL;
  for (let i = 0; i <= 600; i++) {
    const t = i / 600, base = 3.84 + 2.2 * Math.sin(2 * Math.PI * (t - 0.25)) * 1.0;
    const v = base + N * cmNoise[i] * 2.2;
    vin.push([t, v]);
    const a = v > Vref ? VOH : VOL; if (a !== p1 && i) k1++; p1 = a; o1.push([t, a]);
    // Schmitt no inversor: la señal entra por R1, Vref en la entrada −
    if (s2 < 0 && v > V1) { s2 = VOH; k2++; } else if (s2 > 0 && v < V2) { s2 = VOL; k2++; }
    o2.push([t, s2]);
  }
  const hh = (h - 20) / 3, m = { l: 34, r: 90, t: 10, b: 6 };
  g.fillStyle = css('--card'); g.fillRect(0, 0, w, h);
  const panel = (k, y0, y1, title) => {
    g.save(); g.translate(0, k * hh); const r = axes(g, w, hh, m, 0, 1, y0, y1, '', title, y0 < 0 ? [-10, 0, 10] : [2, 4, 6]); g.restore();
    return { X: r.X, Y: (y) => r.Y(y) + k * hh };
  };
  let P = panel(0, 0.5, 7.5, 'v_I');
  g.fillStyle = 'rgba(242,163,58,.18)'; g.fillRect(m.l, P.Y(Math.min(V1, 7.5)), w - m.l - m.r, P.Y(Math.max(V2, 0.5)) - P.Y(Math.min(V1, 7.5)));
  hline(g, Vref, m.l, w - m.r, P.Y, css('--amber'), 'V_ref');
  line(g, vin, P.X, P.Y, css('--blue'), 1.6);
  P = panel(1, -12, 12, 'comparador'); line(g, o1, P.X, P.Y, css('--red'), 1.8);
  g.fillStyle = css('--red'); g.fillText(k1 + ' conmut.', w - m.r + 6, P.Y(0) + 4);
  P = panel(2, -12, 12, 'Schmitt'); line(g, o2, P.X, P.Y, css('--green'), 1.8);
  g.fillStyle = css('--green'); g.fillText(k2 + ' conmut.', w - m.r + 6, P.Y(0) + 4);
  $('cmVH').textContent = fmt(VH) + ' V'; $('cmK1').textContent = k1; $('cmK2').textContent = k2;
  $('cmVals').innerHTML = 'Valores actuales: R<sub>1</sub> = 10 kΩ · R<sub>F</sub> = ' + fmt(RF / 1e3, 0) + ' kΩ → V<sub>1</sub> = ' + fmt(V1) + ' V, V<sub>2</sub> = ' + fmt(V2) + ' V' + (V1 > 6.04 ? ' · <span class="lose">V<sub>1</sub> supera el pico de la señal: el Schmitt nunca conmuta (demasiada histéresis)</span>' : '');
}

// ================= DAC =================
let bits = [0, 0, 0];
document.querySelectorAll('.bit').forEach(b => b.addEventListener('click', () => { const k = +b.dataset.b; bits[k] ^= 1; b.textContent = bits[k]; b.classList.toggle('on', !!bits[k]); dacCalc(); }));
bindRange('dacRF', v => fmt(v, 1) + ' kΩ', dacCalc);
function dacOut(N, RF){ const b2 = (N >> 2) & 1, b1 = (N >> 1) & 1, b0 = N & 1; const v1 = -RF * (5 * b2 / 10 + 5 * b1 / 20 + 5 * b0 / 40); return Math.max(VOL, v1); }
function dacCalc(){
  const RF = +$('dacRF').value, N = bits[2] * 4 + bits[1] * 2 + bits[0];
  const v1 = dacOut(N, RF), vo = -v1;
  $('dacV').textContent = fmt(vo, 3) + ' V'; $('dacN').textContent = N + ' (' + bits[2] + bits[1] + bits[0] + ')';
  $('dacV1').textContent = fmt(v1, 3) + ' V' + (v1 <= VOL ? ' (saturado)' : ''); $('dacL').textContent = fmt(RF * 5 / 40, 3) + ' V';
  $('dacI').textContent = fmt(Math.abs(v1) / RF, 3) + ' mA';
  $('dacVals').innerHTML = 'Valores actuales: b<sub>2</sub>b<sub>1</sub>b<sub>0</sub> = ' + bits[2] + bits[1] + bits[0] + ' · R<sub>F</sub> = ' + fmt(RF, 1) + ' kΩ → v<sub>O1</sub> = ' + fmt(v1, 3) + ' V, v<sub>O</sub> = ' + fmt(vo, 3) + ' V';
  const { g, w, h } = fit($('dacCan')); const m = { l: 34, r: 14, t: 16, b: 24 };
  const { X, Y } = axes(g, w, h, m, -0.5, 7.5, 0, 12, '', 'v_O (V)', [0, 2, 4, 6, 8, 10, 12]);
  for (let k = 0; k < 8; k++) {
    const v = -dacOut(k, RF); g.fillStyle = k === N ? css('--teal') : css('--off');
    g.fillRect(X(k - 0.35), Y(v), X(k + 0.35) - X(k - 0.35), Y(0) - Y(v));
    g.fillStyle = css('--muted'); g.fillText(k.toString(2).padStart(3, '0'), X(k) - 10, h - 8);
  }
  hline(g, VOH, m.l, w - m.r, Y, css('--red'));
}

// ================= RESTADOR / MULTIENTRADAS =================
bindRange('rsI', v => fmt(v, 2) + ' A', rsCalc); bindRange('rsT', v => fmt(v, 1) + ' %', rsCalc); bindRange('rsCM', v => fmt(v, 2) + ' V', rsCalc);
function rsCalc(){
  const I = +$('rsI').value, e = +$('rsT').value / 100, vcm = +$('rsCM').value, RS = 0.1;
  const v1 = vcm, v2 = vcm + RS * I;
  const R1 = 10e3 * (1 + e), R2 = 165e3 * (1 - e), R3 = 10e3 * (1 - e), R4 = 165e3 * (1 + e);
  const ideal = 16.5 * RS * I;
  let real = v2 * R4 / (R3 + R4) * (1 + R2 / R1) - v1 * R2 / R1;
  real = Math.max(VOL, Math.min(VOH, real));
  const Ad = 16.5, Acm = Math.abs(R4 / (R3 + R4) * (1 + R2 / R1) - R2 / R1);
  $('rsVi').textContent = fmt(ideal, 3) + ' V'; $('rsVr').textContent = fmt(real, 3) + ' V';
  $('rsE').textContent = fmt(real - ideal, 3) + ' V'; $('rsC').textContent = Acm > 1e-9 ? fmt(20 * Math.log10(Ad / Acm), 1) + ' dB' : '∞ (ideal)';
  const { g, w, h } = fit($('rsCan')); g.fillStyle = css('--card'); g.fillRect(0, 0, w, h);
  const X = (v) => 10 + v / 4.2 * (w - 20);
  g.fillStyle = css('--bg'); g.fillRect(10, 20, w - 20, 22);
  g.fillStyle = css('--teal'); g.fillRect(10, 20, Math.max(0, X(Math.min(real, 4.2)) - 10), 22);
  g.strokeStyle = css('--red'); g.lineWidth = 2; g.beginPath(); g.moveTo(X(3.3), 14); g.lineTo(X(3.3), 48); g.stroke();
  g.fillStyle = css('--muted'); g.font = '12px Calibri, sans-serif'; g.fillText('0 V', 10, 62); g.fillText('3,3 V (tope del ADC)', X(3.3) - 50, 62);
}
['u1', 'u2', 'u3'].forEach(id => bindRange(id, v => fmt(v, 2) + ' V', uCalc));
function uCalc(){
  const v1 = +$('u1').value, v2 = +$('u2').value, v3 = +$('u3').value;
  const vp = (2 * v1 + v3) / 4, vc = 2 * v1 - 3 * v2 + v3, vo = Math.max(VOL, Math.min(VOH, vc));
  $('uVp').textContent = fmt(vp, 3) + ' V'; $('uVc').textContent = fmt(vc, 3) + ' V'; $('uVo').textContent = fmt(vo, 3) + ' V';
  $('uSat').innerHTML = vc !== vo ? '<span class="lose">Saturación:</span> la salida no puede superar ±10,5 V; reduzca las entradas.' : 'Operación lineal: el corto virtual hace v<sub>−</sub> = v<sub>+</sub> = ' + fmt(vp, 3) + ' V.';
}

// ================= TERMOSTATO =================
let tm;
function tmReset(){ tm = { t: 0, A: { T: 22, Tm: 22, on: 1 }, B: { T: 22, Tm: 22 }, sw: 0, hA: [], hB: [] }; }
tmReset(); $('tmReset').addEventListener('click', tmReset);
bindRange('tmSP', v => fmt(v, 1) + ' °C'); bindRange('tmH', v => '±' + fmt(v / 2, 2) + ' °C', () => {
  const VH = +$('tmH').value * 0.1; $('tmRF').textContent = VH > 0 ? fmt(21 / VH - 1, 0) + ' kΩ' : '∞ (sin histéresis)'; });
bindRange('tmK', v => fmt(v, 2) + ' /°C'); bindRange('tmN', v => fmt(v, 2) + ' °C');
function gauss(){ return (Math.random() + Math.random() + Math.random() - 1.5) * 1.4; }
function tmStep(dt){
  const sp = +$('tmSP').value, H = +$('tmH').value / 2, K = +$('tmK').value, N = +$('tmN').value, tau = 120, dT = 30, Ta = 22;
  const n = Math.ceil(dt / 0.25), h = dt / n;
  for (let i = 0; i < n; i++) {
    const A = tm.A, B = tm.B;
    A.Tm += (A.T - A.Tm) * h / 15; B.Tm += (B.T - B.Tm) * h / 15;
    const meas = A.Tm + N * gauss();
    if (A.on && meas > sp + H) { A.on = 0; tm.sw++; } else if (!A.on && meas < sp - H) { A.on = 1; tm.sw++; }
    const u = Math.max(0, Math.min(1, K * (sp - (B.Tm + N * gauss()))));
    A.T += h / tau * (Ta + dT * A.on - A.T); B.T += h / tau * (Ta + dT * u - B.T);
    tm.t += h;
  }
  tm.hA.push([tm.t, tm.A.T]); tm.hB.push([tm.t, tm.B.T]);
  while (tm.hA.length && tm.hA[0][0] < tm.t - 900) { tm.hA.shift(); tm.hB.shift(); }
}
function tmVals(){
  const sp = +$('tmSP').value, H = +$('tmH').value / 2, R1 = 1;
  const V1 = (sp + H) * 0.1, V2 = (sp - H) * 0.1, VH = V1 - V2;
  if (VH <= 0) { $('tmVals').innerHTML = 'Valores actuales: sin histéresis el circuito es un comparador simple (R<sub>F</sub> → ∞).'; return; }
  const RF = R1 * ((VOH - VOL) / VH - 1), V = (V1 * (RF + R1) - VOH * R1) / RF;
  $('tmVals').innerHTML = 'Valores actuales: umbrales ' + fmt(V2, 3) + ' V y ' + fmt(V1, 3) + ' V → R<sub>F</sub> = ' + fmt(RF, 1) + ' kΩ, V = ' + fmt(V, 3) + ' V, R<sub>A</sub> = ' + fmt(12 / V, 2) + ' kΩ, R<sub>B</sub> = ' + fmt(12 / (12 - V), 2) + ' kΩ';
}
function tmDraw(){
  const { g, w, h } = fit($('tmCan')); const sp = +$('tmSP').value, H = +$('tmH').value / 2;
  const m = { l: 34, r: 14, t: 18, b: 22 }, t1 = Math.max(900, tm.t);
  const { X, Y } = axes(g, w, h, m, t1 - 900, t1, 20, 54, 'tiempo (s)', 'T (°C)', [20, 25, 30, 35, 40, 45, 50]);
  g.fillStyle = 'rgba(242,163,58,.16)'; g.fillRect(m.l, Y(sp + H), w - m.l - m.r, Math.max(1, Y(sp - H) - Y(sp + H)));
  hline(g, sp, m.l, w - m.r, Y, css('--muted'));
  line(g, tm.hA, X, Y, css('--red'), 2); line(g, tm.hB, X, Y, css('--blue'), 2);
  $('tmSw').textContent = tm.sw; $('tmT1').textContent = fmt(tm.A.T, 1) + ' °C'; $('tmT2').textContent = fmt(tm.B.T, 1) + ' °C';
  tmVals(); $('tmState').textContent = 'Calefactor ON/OFF: ' + (tm.A.on ? 'encendido' : 'apagado');
}

// ================= JUEGO =================
let jg = { t: 0, hits: 0, tries: 0, streak: 0, latch: false, flash: 0, mark: null };
bindRange('jgR', v => v + ' kΩ', jgCalc); bindRange('jgW', v => fmt(v, 2) + ' V', jgCalc);
function jgF(){ return 2.1 / (4 * (+$('jgR').value * 1e3) * 1e-6); }
function jgCalc(){ const f = jgF(), W = +$('jgW').value, slope = 4 * 5 * f; $('jgF').textContent = fmt(f, 2) + ' Hz'; $('jgTw').textContent = fmt(2 * W / slope * 1000, 0) + ' ms ×2 por periodo';
  $('jgVals').innerHTML = 'Valores actuales: R = ' + $('jgR').value + ' kΩ, C = 1 µF, R<sub>1</sub> = 10 kΩ, R<sub>F</sub> = 21 kΩ → f = ' + fmt(f, 2) + ' Hz · ventana ±' + fmt(W, 2) + ' V'; }
function tri(t){ const f = jgF(), ph = (t * f) % 1; return ph < 0.5 ? -5 + 20 * ph : 15 - 20 * ph; }
function jgPress(){
  const v = tri(jg.t), W = +$('jgW').value, inWin = Math.abs(v) < W, vW = inWin ? 10.5 : 0;
  jg.tries++; jg.mark = { v, ok: inWin, t: jg.t };
  if (vW > 1.05) { jg.latch = true; jg.hits++; jg.streak++; $('jgMsg').innerHTML = '<span class="win">¡Ganaste!</span> v = ' + fmt(v, 2) + ' V'; }
  else { jg.streak = 0; $('jgMsg').innerHTML = '<span class="lose">Fuera de la ventana</span> v = ' + fmt(v, 2) + ' V'; }
  $('jgLed').classList.toggle('on', jg.latch);
  $('jgS').textContent = jg.hits + ' / ' + jg.tries; $('jgSt').textContent = jg.streak;
}
$('jgBtn').addEventListener('click', jgPress);
$('jgRst').addEventListener('click', () => { jg.latch = false; $('jgLed').classList.remove('on'); $('jgMsg').textContent = 'Biestable reiniciado (−12 V)'; });
document.addEventListener('keydown', (e) => { if (active === 'juego' && (e.code === 'Space' || e.key === ' ')) { e.preventDefault(); jgPress(); } });
$('jgS').textContent = '0 / 0'; $('jgSt').textContent = '0';
function jgDraw(){
  const { g, w, h } = fit($('jgCan')); const W = +$('jgW').value; const m = { l: 34, r: 14, t: 14, b: 22 };
  const span = 2.5, t1 = jg.t, t0 = t1 - span;
  const { X, Y } = axes(g, w, h, m, t0, t1 + 0.4, -6, 6, 'tiempo (s)', 'v (V)', [-5, 0, 5]);
  g.fillStyle = 'rgba(46,155,87,.22)'; g.fillRect(m.l, Y(W), w - m.l - m.r, Y(-W) - Y(W));
  const pts = []; for (let k = 0; k <= 300; k++) { const t = t0 + span * k / 300; pts.push([t, tri(t)]); }
  line(g, pts, X, Y, css('--blue'), 2.2);
  const v = tri(t1), inWin = Math.abs(v) < W;
  g.fillStyle = inWin ? css('--green') : css('--amber'); g.beginPath(); g.arc(X(t1), Y(v), 9, 0, 7); g.fill();
  if (jg.mark && jg.t - jg.mark.t < span) { g.strokeStyle = jg.mark.ok ? css('--green') : css('--red'); g.lineWidth = 3; g.beginPath(); g.arc(X(jg.mark.t), Y(jg.mark.v), 13, 0, 7); g.stroke(); }
  g.fillStyle = css('--muted'); g.fillText('ventana ±' + fmt(W, 2) + ' V', w - m.r - 110, Y(W) - 6);
}

// ================= THEREMIN =================
let ac = null, osc = null, gain = null, thLight = 50;
const thL = bindRange('thL', v => v + ' %', () => { thLight = +thL.value; thCalc(); });
function thR(){ return 1e3 * Math.pow(10, 3 - thLight / 100 * 2.3); } // 1 MΩ (oscuro) … 5 kΩ (luz)
function thFreq(){ return 27e3 / (4 * (thR() + 4.7e3) * 47e-9 * 10e3); }
function thCalc(){
  const R = thR(), f = thFreq();
  $('thR').textContent = R >= 1e6 ? fmt(R / 1e6, 2) + ' MΩ' : fmt(R / 1e3, 1) + ' kΩ';
  $('thF').textContent = f >= 1000 ? fmt(f / 1000, 2) + ' kHz' : fmt(f, 0) + ' Hz';
  if (osc) osc.frequency.setTargetAtTime(f, ac.currentTime, 0.03);
  $('thVals').innerHTML = 'Valores actuales: R<sub>LDR</sub> + 4,7 kΩ = ' + fmt((R + 4.7e3) / 1e3, 1) + ' kΩ, C = 47 nF, R<sub>1</sub> = 10 kΩ, R<sub>F</sub> = 27 kΩ → f = ' + fmt(f, 0) + ' Hz';
}
$('thOn').addEventListener('click', () => {
  try {
    if (!ac) { const A = window.AudioContext || window.webkitAudioContext; ac = new A(); }
    if (osc) { osc.stop(); osc = null; $('thOn').textContent = 'Activar sonido'; return; }
    ac.resume(); osc = ac.createOscillator(); gain = ac.createGain(); osc.type = 'square'; gain.gain.value = 0.06;
    osc.connect(gain).connect(ac.destination); osc.frequency.value = thFreq(); osc.start(); $('thOn').textContent = 'Silenciar';
  } catch (e) { $('thOn').textContent = 'Audio no disponible'; }
});
const thc = $('thCan');
function thPointer(e){ const r = thc.getBoundingClientRect(); const y = (e.clientY - r.top) / r.height; thLight = Math.round(Math.max(0, Math.min(1, 1 - y)) * 100); thL.value = thLight; $('thLO').textContent = thLight + ' %'; thCalc(); }
thc.addEventListener('pointermove', thPointer); thc.addEventListener('pointerdown', thPointer);
let thPh = 0;
function thDraw(dt){
  const { g, w, h } = fit(thc); const L = thLight / 100;
  const gr = g.createLinearGradient(0, 0, 0, h); gr.addColorStop(0, '#FFF3C4'); gr.addColorStop(1, '#1B2A30'); g.fillStyle = gr; g.fillRect(0, 0, w, h);
  const y = (1 - L) * h; g.fillStyle = 'rgba(242,163,58,.9)'; g.beginPath(); g.arc(w / 2, y, 16, 0, 7); g.fill();
  g.fillStyle = '#12343F'; g.font = '13px Calibri, sans-serif'; g.fillText('mucha luz', 10, 18); g.fillStyle = '#E6EEF1'; g.fillText('sombra', 10, h - 10);
  const c = fit($('thW')); const f = thFreq(); thPh += dt * Math.min(f, 6);
  c.g.fillStyle = css('--card'); c.g.fillRect(0, 0, c.w, c.h);
  const cyc = Math.max(1, Math.min(30, f / 60));
  const tp = [], sq = [];
  for (let k = 0; k <= 400; k++) { const x = k / 400, ph = (x * cyc + thPh) % 1; tp.push([x, ph < 0.5 ? -1 + 4 * ph : 3 - 4 * ph]); sq.push([x, ph < 0.5 ? 1.6 : -1.6]); }
  const X = (x) => 6 + x * (c.w - 12), Y = (v) => c.h / 2 - v * c.h / 4.2;
  line(c.g, sq, X, Y, css('--off'), 1.5); line(c.g, tp, X, Y, css('--teal'), 2);
}

// ---------- ciclo principal ----------
function resizeAll(){ sfDraw(); cmDraw(); dacCalc(); rsCalc(); uCalc(); jgCalc(); thCalc(); }
let last = performance.now();
function loop(now){
  const dt = Math.min(0.1, (now - last) / 1000); last = now;
  sfStep(dt * +$('sfSp').value); tmStep(dt * 20); jg.t += dt;
  if (active === 'semaforo') sfDraw();
  if (active === 'termo') tmDraw();
  if (active === 'juego') jgDraw();
  if (active === 'theremin') thDraw(dt);
  requestAnimationFrame(loop);
}
window.addEventListener('resize', resizeAll);
sfCalc(); resizeAll(); $('tmH').dispatchEvent(new Event('input'));
requestAnimationFrame(loop);
})();
</script>
</body>
</html>
'''

import html as _html
from IPython.display import HTML, display
with open('operacionales_semana3.html', 'w', encoding='utf-8') as f:
    f.write(PAGINA)
display(HTML('<iframe srcdoc="' + _html.escape(PAGINA, quote=True) + '" style="width:100%;height:900px;border:1px solid #D5DEE2;border-radius:12px" scrolling="yes" allow="autoplay"></iframe>'))

In [ ]:
# Opcional: descargar la página para abrirla en cualquier navegador
try:
    from google.colab import files
    files.download('operacionales_semana3.html')
except ImportError:
    print('Fuera de Colab: el archivo quedó guardado en la carpeta de trabajo.')

## 6. Ejercicios propuestos
1. Rediseñe el semáforo para **8 s de verde, 2 s de amarillo y 5 s de rojo**. Calcule $R_u$, $R_a$, $R_b$ y $R_c$ y compruébelo con el semáforo interactivo.
2. Agregue un **cuarto comparador** que haga parpadear el verde durante el último segundo (pista: compare la rampa con un umbral $V_A - 0{,}66\,\mathrm{V}$).
3. Diseñe un D/A de **4 bits** con paso de 0,5 V. ¿Qué resistencia debe tener el bit menos significativo?
4. Diseñe $v_O = 4v_1 - v_2$ con `disenar_multientradas` y verifique el balance eléctrico.
5. Cambie el termostato para encender a 37 °C y apagar a 38 °C (incubadora). ¿Cuánto valen $R_F$, $R_A$ y $R_B$?